In [1]:
# EquiBind Batch Docking Pipeline with Multiple Pose Generation
#
# This notebook docks all protein-ligand combinations using EquiBind,
# generating multiple diverse poses per combination using RDKit conformer generation.
#
# Strategy: Generate multiple 3D conformers with RDKit's EmbedMultipleConfs(),
# then run EquiBind on each conformer to produce diverse docked poses.
#
# Uses conda environment 'equibind' to run multiligand_inference.py:
#   conda run -n equibind python ~/docking_tools/EquiBind/multiligand_inference.py \
#     -o ./equibind_out -r protein.pdb -l ligand.sdf --device cpu

# Imports

In [2]:
from __future__ import annotations

import csv
import hashlib
import itertools
import json
import os
import shutil
import subprocess
import time
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from enum import IntEnum
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolAlign

# Suppress RDKit deprecation warnings (e.g. GetValence)
RDLogger.DisableLog('rdApp.*')

# Config

In [3]:

# ── Configuration ──────────────────────────────────────────────────────────

# Paths
workspace_root = Path.cwd()
drugs_dir = workspace_root / "Drugs"
receptors_dir = workspace_root / "Orai"

# Pocket result directories (from previous runs)
fpocket_results_folder = workspace_root / "fpocket_results"
p2rank_folder = workspace_root / "p2rank_results"

# EquiBind paths
EQUIBIND_DIR = Path("/home/manndo/docking_tools/EquiBind")
EQUIBIND_MULTILIGAND_SCRIPT = EQUIBIND_DIR / "multiligand_inference.py"
EQUIBIND_DEVICE = "cuda"  # "cpu" or "cuda"

# Output directories
POCKET_GUIDED_OUTPUT_DIR = workspace_root / "equibind_pocket_guided"
POCKET_GUIDED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EQUIBIND_BATCH_DIR = workspace_root / "equibind_batches"
EQUIBIND_OUTPUT_DIR = workspace_root / "equibind_docked_poses"
EQUIBIND_BATCH_DIR.mkdir(parents=True, exist_ok=True)
EQUIBIND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = POCKET_GUIDED_OUTPUT_DIR / "wdlogs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Pocket-guided docking settings
N_TOP_POCKETS: int = 5          # Top-ranked pockets to use from each method
POSES_PER_POCKET: int = 3       # Poses to generate per pocket (using different conformers)
N_UNGUIDED_POSES: int = 5      # Natural EquiBind poses per protein-ligand pair
POCKET_MATCH_THRESHOLD: float = 8.0  # Distance (Å) to consider a pose "inside" a pocket

# Pocket enforcement settings
FORCE_POCKET: bool = True        # If True, reject poses whose centroid drifts outside the pocket
MAX_POCKET_TRIALS: int = 200      # Max conformer trials per desired pose when FORCE_POCKET is True
CLAMP_POSE_TO_POCKET: bool = True  # If True, clamp drifted poses back to pocket (instead of rejecting)

# Protein cropping settings (constrains EquiBind's search space to the pocket region)
USE_PROTEIN_CROPPING: bool = True   # If True, dock against cropped protein (pocket only)
POCKET_CROP_RADIUS: float = 6.0    # Residues within this radius of pocket center are kept
POCKET_CROP_BUFFER: float = 3.0     # Additional buffer for context

# Batch docking settings
NUM_POSES: int = 30             # Unique poses per protein-ligand combination
NUM_CONFORMERS: int = 10        # RDKit conformers to generate (>= NUM_POSES for diversity)
MAX_ATTEMPTS: int = NUM_POSES * 30  # Fallback if conformers fail
POSE_RMSD_THRESHOLD: float = 1.0    # Min RMSD (Å) between poses to consider distinct

# RDKit conformer generation parameters
RDKIT_SEEDS: list = [42, 123, 456, 789, 1001, 2022, 3141, 5926, 8675, 9999]
CONFORMERS_PER_SEED: int = 5
RDKIT_RANDOM_SEED: int = 42
RDKIT_NUM_THREADS: int = 0              # 0 = use all available threads
RDKIT_PRUNE_RMS_THRESH: float = 0.2     # Prune similar conformers during generation

# Overwrite / skip settings
SKIP_EXISTING: bool = False
OVERWRITE_EXISTING: bool = True  # Set to True to re-dock existing combinations

# Check for reduce executable (for adding hydrogens to proteins)
REDUCE_EXECUTABLE = shutil.which("reduce")

# ── Validate ───────────────────────────────────────────────────────────────
if not drugs_dir.exists():
    raise FileNotFoundError(f"Ligand directory missing: {drugs_dir}")
if not receptors_dir.exists():
    raise FileNotFoundError(f"Receptor directory missing: {receptors_dir}")

print("=" * 80)
print("EquiBind Batch Docking Configuration")
print("=" * 80)
print(f"Number of poses per combination: {NUM_POSES}")
print(f"RDKit conformers to generate:    {NUM_CONFORMERS}")
print(f"Maximum attempts per combination:{MAX_ATTEMPTS}")
print(f"Pose RMSD threshold:             {POSE_RMSD_THRESHOLD} Å")
print(f"RDKit prune RMS threshold:       {RDKIT_PRUNE_RMS_THRESH} Å")
print(f"EquiBind script:  {EQUIBIND_MULTILIGAND_SCRIPT}")
print(f"EquiBind device:  {EQUIBIND_DEVICE}")
print(f"Batch directory:  {EQUIBIND_BATCH_DIR}")
print(f"Output directory: {EQUIBIND_OUTPUT_DIR}")
print()

if not EQUIBIND_DIR.exists():
    print(f"⚠️  WARNING: EquiBind directory not found at {EQUIBIND_DIR}")
    print("   Please update EQUIBIND_DIR to point to your EquiBind installation")
elif not EQUIBIND_MULTILIGAND_SCRIPT.exists():
    print(f"⚠️  WARNING: multiligand_inference.py not found at {EQUIBIND_MULTILIGAND_SCRIPT}")
else:
    print(f"✓ EquiBind script found")

EquiBind Batch Docking Configuration
Number of poses per combination: 30
RDKit conformers to generate:    10
Maximum attempts per combination:900
Pose RMSD threshold:             1.0 Å
RDKit prune RMS threshold:       0.2 Å
EquiBind script:  /home/manndo/docking_tools/EquiBind/multiligand_inference.py
EquiBind device:  cuda
Batch directory:  /home/manndo/master_dev/equibind_batches
Output directory: /home/manndo/master_dev/equibind_docked_poses

✓ EquiBind script found


# Signal Monitor

In [4]:
# ── Signal Monitor ──────────────────────────────────────────────────────────
# Three verbosity levels control what gets printed during docking.
#
#   MonitorLevel.ALL      → Everything: EquiBind invocations, returned poses,
#                           pocket distance checks, conformer generation, etc.
#   MonitorLevel.WARNING  → Rejected poses, dock failures, pocket drift,
#                           plus everything from CRITICAL.
#   MonitorLevel.CRITICAL → Only fatal errors (timeouts, missing output, total
#                           combo failures).
# ────────────────────────────────────────────────────────────────────────────


class MonitorLevel(IntEnum):
    """Signal monitoring verbosity levels."""
    ALL = 1          # Full trace
    WARNING = 2      # Warnings + critical only
    CRITICAL = 3     # Fatal errors only

# ── Set the active monitoring level here ──────────────────────────────────
# Change this to MonitorLevel.WARNING or MonitorLevel.CRITICAL to reduce output.
MONITOR_LEVEL = MonitorLevel.ALL


class DockingMonitor:
    """Structured signal monitor for the EquiBind docking pipeline.

    Every message is tagged with a level; only messages at or above the
    configured threshold are printed.  Colour prefixes make it easy to
    scan terminal output.
    """

    _PREFIXES = {
        MonitorLevel.ALL:      "    ·",
        MonitorLevel.WARNING:  " ⚠️ ",
        MonitorLevel.CRITICAL: " 🔴",
    }

    def __init__(self, level: MonitorLevel = MonitorLevel.ALL):
        self.level = level
        self.call_count = 0
        self.success_count = 0
        self.fail_count = 0
        self.rejected_count = 0
        self.in_pocket_count = 0
        self.outside_pocket_count = 0

    # ── core dispatcher ──────────────────────────────────────────────────
    def _emit(self, lvl: MonitorLevel, msg: str) -> None:
        if lvl >= self.level:
            prefix = self._PREFIXES.get(lvl, "")
            print(f"{prefix} {msg}")

    # ── convenience loggers ──────────────────────────────────────────────
    def info(self, msg: str) -> None:
        """Verbose trace (ALL level)."""
        self._emit(MonitorLevel.ALL, msg)

    def warning(self, msg: str) -> None:
        """Non-fatal issue (WARNING level)."""
        self._emit(MonitorLevel.WARNING, msg)

    def critical(self, msg: str) -> None:
        """Fatal / blocking error (CRITICAL level)."""
        self._emit(MonitorLevel.CRITICAL, msg)

    # ── EquiBind call / return ───────────────────────────────────────────
    def equibind_call(self, protein: str, ligand: str, output_dir: str,
                      seed: int, device: str) -> None:
        """Log an outgoing EquiBind invocation."""
        self.call_count += 1
        self.info(
            f"[CALL #{self.call_count}] EquiBind ← "
            f"protein={protein}, ligand={ligand}, "
            f"seed={seed}, device={device}, out={output_dir}"
        )

    def equibind_return(self, success: bool, output_sdf: Optional[str],
                        error: str = "") -> None:
        """Log what EquiBind returned."""
        if success:
            self.success_count += 1
            self.info(f"[RECV] EquiBind → OK  output={output_sdf}")
        else:
            self.fail_count += 1
            self.warning(f"[RECV] EquiBind → FAIL  error={error[:200]}")

    # ── Pocket proximity ─────────────────────────────────────────────────
    def pose_accepted_in_pocket(self, pose_id: str, pocket_id: str,
                                distance: float, threshold: float) -> None:
        """Log a pose that landed inside the target pocket."""
        self.in_pocket_count += 1
        self.info(
            f"[POCKET ✓] {pose_id} INSIDE {pocket_id}  "
            f"(dist={distance:.1f}Å ≤ {threshold:.1f}Å)"
        )

    def pose_rejected_from_pocket(self, pose_id: str, pocket_id: str,
                                  distance: float, threshold: float) -> None:
        """Log a pose that drifted outside the target pocket."""
        self.rejected_count += 1
        self.warning(
            f"[POCKET ✗] {pose_id} OUTSIDE {pocket_id}  "
            f"(dist={distance:.1f}Å > {threshold:.1f}Å) → REJECTED"
        )

    def pose_outside_all_pockets(self, pose_id: str,
                                 nearest_pocket: str,
                                 nearest_dist: float,
                                 threshold: float) -> None:
        """Log an unguided pose that is not near any known pocket."""
        self.outside_pocket_count += 1
        self.warning(
            f"[POCKET ✗] {pose_id} not in any pocket  "
            f"(nearest={nearest_pocket}, dist={nearest_dist:.1f}Å > {threshold:.1f}Å)"
        )

    def pose_inside_pocket(self, pose_id: str, pocket_id: str,
                           distance: float, threshold: float,
                           pocket_source: str) -> None:
        """Log an unguided pose that falls inside a known pocket."""
        self.in_pocket_count += 1
        self.info(
            f"[POCKET ✓] {pose_id} in {pocket_source} pocket {pocket_id}  "
            f"(dist={distance:.1f}Å ≤ {threshold:.1f}Å)"
        )

    # ── Section headers ──────────────────────────────────────────────────
    def header(self, msg: str) -> None:
        """Always printed regardless of level."""
        print(f"\n{'─' * 70}")
        print(f"  {msg}")
        print(f"{'─' * 70}")

    def section(self, msg: str) -> None:
        """Printed at ALL level."""
        self._emit(MonitorLevel.ALL, f"── {msg} ──")

    # ── Summary ──────────────────────────────────────────────────────────
    def print_summary(self) -> None:
        """Print accumulated counters (always shown)."""
        print(f"\n{'═' * 70}")
        print("  SIGNAL MONITOR SUMMARY")
        print(f"{'═' * 70}")
        print(f"  EquiBind calls:        {self.call_count}")
        print(f"  Successful returns:    {self.success_count}")
        print(f"  Failed returns:        {self.fail_count}")
        print(f"  Poses inside pocket:   {self.in_pocket_count}")
        print(f"  Poses rejected/outside:{self.rejected_count + self.outside_pocket_count}")
        print(f"    ├ guided rejected:   {self.rejected_count}")
        print(f"    └ unguided outside:  {self.outside_pocket_count}")
        print(f"{'═' * 70}")


# Instantiate the global monitor using the configured level
monitor = DockingMonitor(level=MONITOR_LEVEL)
print(f"DockingMonitor initialised  —  level = {MONITOR_LEVEL.name}")
print(f"  ALL      = full trace of every call, return, and pocket check")
print(f"  WARNING  = only rejected poses, failures, and pocket drift")
print(f"  CRITICAL = only fatal errors")

DockingMonitor initialised  —  level = ALL
  ALL      = full trace of every call, return, and pocket check
  WARNING  = only rejected poses, failures, and pocket drift
  CRITICAL = only fatal errors


# Validate GPU Support

In [5]:
import torch
import subprocess

# ============================================================================
# VALIDATE GPU SUPPORT
# ============================================================================


print("=" * 80)
print("GPU SUPPORT VALIDATION")
print("=" * 80)

# Check PyTorch CUDA availability
print("\nPyTorch CUDA Status:")
print(f"  PyTorch version: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  CUDA version: {torch.version.cuda}")
    print(f"  Number of GPUs: {torch.cuda.device_count()}")
    print(f"  Current GPU: {torch.cuda.current_device()}")
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Test GPU with a simple tensor operation
    try:
        x = torch.randn(100, 100).cuda()
        y = torch.randn(100, 100).cuda()
        z = torch.matmul(x, y)
        print(f"  ✓ GPU tensor operations working")
    except Exception as e:
        print(f"  ✗ GPU tensor test failed: {e}")
else:
    print("  ⚠️  No CUDA-capable GPU detected")
    print("  EquiBind will use CPU (slower)")

# Check nvidia-smi
print("\nNVIDIA GPU Information (nvidia-smi):")
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(result.stdout)
    else:
        print("  ⚠️  nvidia-smi not available or failed")
except FileNotFoundError:
    print("  ⚠️  nvidia-smi command not found")
except Exception as e:
    print(f"  ⚠️  Error running nvidia-smi: {e}")

# Recommendation
print("\n" + "=" * 80)
if torch.cuda.is_available():
    print("✓ GPU ACCELERATION AVAILABLE")
    print(f"  Recommended setting: EQUIBIND_DEVICE = 'cuda'")
    print(f"  Current setting: EQUIBIND_DEVICE = '{EQUIBIND_DEVICE}'")
else:
    print("⚠️  GPU NOT AVAILABLE - Using CPU")
    print(f"  Recommended setting: EQUIBIND_DEVICE = 'cpu'")
    print(f"  Current setting: EQUIBIND_DEVICE = '{EQUIBIND_DEVICE}'")
print("=" * 80)

GPU SUPPORT VALIDATION

PyTorch CUDA Status:
  PyTorch version: 2.1.0+cu118
  CUDA available: True
  CUDA version: 11.8
  Number of GPUs: 1
  Current GPU: 0
  GPU Name: NVIDIA GeForce RTX 4070 Laptop GPU
  GPU Memory: 8.59 GB
  ✓ GPU tensor operations working

NVIDIA GPU Information (nvidia-smi):
Thu Feb 26 11:21:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.07             Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 

# Pocket Cropping

In [6]:
# ============================================================================
# PROTEIN CROPPING FOR POCKET-CONSTRAINED DOCKING
# ============================================================================
# This cell implements protein cropping to constrain EquiBind's search space
# to a specific binding pocket. By extracting only residues within a defined
# radius of the pocket center, EquiBind physically cannot predict binding
# poses outside the pocket region.
#
# Key functions:
#   - crop_protein_to_pocket(): Extract residues near pocket center
#   - get_cropped_protein_for_pocket(): Cache management for cropped proteins
#   - Offset tracking to transform coordinates back to original frame
#
# Configuration is in cell 5:
#   USE_PROTEIN_CROPPING, POCKET_CROP_RADIUS, POCKET_CROP_BUFFER
# ============================================================================

from typing import Set, Dict, Tuple, Optional, List
from pathlib import Path
import numpy as np

# Include all atoms of a residue if any atom is in range
INCLUDE_FULL_RESIDUES: bool = True

# Cache: (protein_path, pocket_unique_id) → (cropped_pdb_path, offset_vector)
_cropped_protein_cache: Dict[Tuple[str, str], Tuple[Path, np.ndarray]] = {}


def _parse_pdb_atoms(pdb_path: Path) -> List[dict]:
    """Parse ATOM/HETATM records from a PDB file.
    
    Returns list of dicts with keys:
        line, record_type, atom_num, atom_name, res_name, chain, res_num, x, y, z
    """
    atoms = []
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                try:
                    atoms.append({
                        'line': line,
                        'record_type': line[0:6].strip(),
                        'atom_num': int(line[6:11]),
                        'atom_name': line[12:16].strip(),
                        'res_name': line[17:20].strip(),
                        'chain': line[21:22].strip() or 'A',
                        'res_num': int(line[22:26]),
                        'x': float(line[30:38]),
                        'y': float(line[38:46]),
                        'z': float(line[46:54]),
                    })
                except (ValueError, IndexError):
                    continue
    return atoms


def _get_residues_near_center(
    atoms: List[dict],
    center: Tuple[float, float, float],
    radius: float,
    include_full_residues: bool = True,
) -> Set[Tuple[str, int]]:
    """Find residues with at least one atom within radius of center.
    
    Returns set of (chain, res_num) tuples.
    """
    center_np = np.array(center)
    near_residues: Set[Tuple[str, int]] = set()
    
    for atom in atoms:
        coord = np.array([atom['x'], atom['y'], atom['z']])
        dist = np.linalg.norm(coord - center_np)
        if dist <= radius:
            near_residues.add((atom['chain'], atom['res_num']))
    
    return near_residues


def crop_protein_to_pocket(
    protein_pdb: Path,
    pocket_center: Tuple[float, float, float],
    output_pdb: Path,
    crop_radius: float = POCKET_CROP_RADIUS,
    buffer: float = POCKET_CROP_BUFFER,
    include_full_residues: bool = INCLUDE_FULL_RESIDUES,
) -> Tuple[bool, np.ndarray, int, int]:
    """Crop a protein PDB to residues near a pocket center.
    
    Args:
        protein_pdb: Input full protein PDB
        pocket_center: (x, y, z) pocket center coordinates
        output_pdb: Output cropped PDB path
        crop_radius: Radius around pocket center to include
        buffer: Additional buffer for context residues
        include_full_residues: If True, include all atoms of selected residues
        
    Returns:
        (success, offset_vector, n_residues_kept, n_atoms_kept)
        
        offset_vector is the translation applied to center the pocket at origin.
        To convert cropped coordinates back to original frame: coord + offset_vector
    """
    effective_radius = crop_radius + buffer
    
    # Parse full protein
    atoms = _parse_pdb_atoms(protein_pdb)
    if not atoms:
        return False, np.zeros(3), 0, 0
    
    # Find residues within range
    near_residues = _get_residues_near_center(
        atoms, pocket_center, effective_radius, include_full_residues
    )
    
    if not near_residues:
        monitor.warning(f"No residues found within {effective_radius}Å of pocket center")
        return False, np.zeros(3), 0, 0
    
    # Filter atoms to keep
    if include_full_residues:
        kept_atoms = [a for a in atoms if (a['chain'], a['res_num']) in near_residues]
    else:
        # Only keep atoms actually within radius
        center_np = np.array(pocket_center)
        kept_atoms = []
        for a in atoms:
            coord = np.array([a['x'], a['y'], a['z']])
            if np.linalg.norm(coord - center_np) <= effective_radius:
                kept_atoms.append(a)
    
    if not kept_atoms:
        return False, np.zeros(3), 0, 0
    
    # Compute offset: we'll translate so pocket center is at origin
    # This helps EquiBind by centering the binding region
    offset = np.array(pocket_center)
    
    # Write cropped PDB with translated coordinates
    output_pdb.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_pdb, 'w', encoding='utf-8') as f:
        f.write(f"REMARK   CROPPED PROTEIN - pocket center at origin\n")
        f.write(f"REMARK   Original pocket center: {pocket_center[0]:.3f} {pocket_center[1]:.3f} {pocket_center[2]:.3f}\n")
        f.write(f"REMARK   Crop radius: {effective_radius:.1f} A\n")
        f.write(f"REMARK   Residues kept: {len(near_residues)}\n")
        f.write(f"REMARK   To restore original coords: add offset ({offset[0]:.3f}, {offset[1]:.3f}, {offset[2]:.3f})\n")
        
        atom_num = 1
        for a in kept_atoms:
            # Translate to center pocket at origin
            new_x = a['x'] - offset[0]
            new_y = a['y'] - offset[1]
            new_z = a['z'] - offset[2]
            
            # Reconstruct PDB line with new coordinates
            line = a['line']
            new_line = (
                f"{line[0:6]}"  # record type
                f"{atom_num:5d}"  # atom number (renumbered)
                f"{line[11:30]}"  # atom name, res name, chain, res num
                f"{new_x:8.3f}{new_y:8.3f}{new_z:8.3f}"  # coordinates
                f"{line[54:]}"  # rest of line (occupancy, B-factor, element)
            )
            f.write(new_line)
            atom_num += 1
        
        f.write("END\n")
    
    n_residues = len(near_residues)
    n_atoms = len(kept_atoms)
    
    return True, offset, n_residues, n_atoms


def get_cropped_protein_for_pocket(
    protein_pdb: Path,
    pocket: 'PocketInfo',  # Forward reference - defined in cell 12
    prep_dir: Path,
    crop_radius: float = POCKET_CROP_RADIUS,
) -> Tuple[Optional[Path], np.ndarray]:
    """Get or create a cropped protein PDB for a specific pocket.
    
    Uses caching to avoid re-cropping for the same protein/pocket combination.
    
    Args:
        protein_pdb: Full protein PDB path
        pocket: PocketInfo object with center coordinates
        prep_dir: Directory to store cropped PDB files
        crop_radius: Radius around pocket center
        
    Returns:
        (cropped_pdb_path, offset_vector) or (None, zeros) on failure
        
        The offset_vector should be added to docked coordinates to get
        original-frame coordinates.
    """
    cache_key = (str(protein_pdb), pocket.unique_id)
    
    if cache_key in _cropped_protein_cache:
        return _cropped_protein_cache[cache_key]
    
    # Create cropped protein
    cropped_pdb = prep_dir / f"{protein_pdb.stem}_crop_{pocket.unique_id}.pdb"
    
    success, offset, n_res, n_atoms = crop_protein_to_pocket(
        protein_pdb=protein_pdb,
        pocket_center=pocket.center,
        output_pdb=cropped_pdb,
        crop_radius=crop_radius,
    )
    
    if success:
        monitor.info(f"Cropped protein for {pocket.unique_id}: {n_res} residues, {n_atoms} atoms "
                     f"(radius={crop_radius + POCKET_CROP_BUFFER:.1f}Å)")
        _cropped_protein_cache[cache_key] = (cropped_pdb, offset)
        return cropped_pdb, offset
    else:
        monitor.warning(f"Failed to crop protein for pocket {pocket.unique_id}")
        return None, np.zeros(3)


def translate_pose_back_to_original_frame(
    sdf_path: Path,
    offset: np.ndarray,
    output_path: Path,
) -> bool:
    """Translate a docked pose back to the original protein coordinate frame.
    
    After docking against a cropped protein (centered at origin), this function
    shifts the ligand coordinates back to the original frame.
    
    Args:
        sdf_path: Input SDF from docking against cropped protein
        offset: The offset vector returned by crop_protein_to_pocket
        output_path: Output SDF in original coordinate frame
        
    Returns:
        True on success, False on failure
    """
    try:
        suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False)
        mol = next(iter(suppl), None)
        if mol is None:
            return False
        
        mol = Chem.RWMol(mol)
        conf = mol.GetConformer()
        
        for i in range(mol.GetNumAtoms()):
            pos = conf.GetAtomPosition(i)
            # Add offset to restore original coordinates
            conf.SetAtomPosition(i, (
                pos.x + offset[0],
                pos.y + offset[1],
                pos.z + offset[2],
            ))
        
        writer = Chem.SDWriter(str(output_path))
        writer.write(mol)
        writer.close()
        
        return output_path.exists() and output_path.stat().st_size > 0
    except Exception as e:
        monitor.warning(f"Failed to translate pose back to original frame: {e}")
        return False


def clamp_pose_to_pocket(
    sdf_path: Path,
    pocket_center: Tuple[float, float, float],
    max_distance: float,
    output_path: Path,
) -> Tuple[bool, float, float]:
    """Clamp a docked pose so its centroid is within max_distance of pocket center.
    
    If the pose centroid is farther than max_distance from the pocket center,
    the entire pose is translated to bring the centroid exactly to max_distance
    from the pocket center (along the line from pocket to current centroid).
    
    This ensures ALL poses stay within the defined pocket region.
    
    Args:
        sdf_path: Input SDF file
        pocket_center: Target pocket center (x, y, z)
        max_distance: Maximum allowed distance from pocket center
        output_path: Output SDF with clamped coordinates
        
    Returns:
        (success, original_distance, new_distance)
    """
    try:
        suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False)
        mol = next(iter(suppl), None)
        if mol is None:
            return False, 0.0, 0.0
        
        mol = Chem.RWMol(mol)
        conf = mol.GetConformer()
        
        # Compute current centroid
        positions = conf.GetPositions()
        centroid = positions.mean(axis=0)
        pocket_np = np.array(pocket_center)
        
        # Compute distance from pocket center
        displacement = centroid - pocket_np
        original_distance = np.linalg.norm(displacement)
        
        if original_distance <= max_distance:
            # Already within bounds, just copy
            shutil.copy2(sdf_path, output_path)
            return True, original_distance, original_distance
        
        # Compute shift needed: move centroid to exactly max_distance from pocket
        # Unit vector from pocket to centroid
        direction = displacement / original_distance
        target_centroid = pocket_np + direction * max_distance
        shift = target_centroid - centroid
        
        # Apply shift to all atoms
        for i in range(mol.GetNumAtoms()):
            pos = conf.GetAtomPosition(i)
            conf.SetAtomPosition(i, (
                pos.x + shift[0],
                pos.y + shift[1],
                pos.z + shift[2],
            ))
        
        # Write clamped SDF
        writer = Chem.SDWriter(str(output_path))
        writer.write(mol)
        writer.close()
        
        new_distance = max_distance  # By construction
        
        return output_path.exists() and output_path.stat().st_size > 0, original_distance, new_distance
        
    except Exception as e:
        monitor.warning(f"Failed to clamp pose to pocket: {e}")
        return False, 0.0, 0.0


def clear_cropped_protein_cache():
    """Clear the cropped protein cache (useful for memory management)."""
    global _cropped_protein_cache
    _cropped_protein_cache.clear()
    monitor.info("Cropped protein cache cleared")


# ── Statistics helper ─────────────────────────────────────────────────────

def estimate_search_space_reduction(
    protein_pdb: Path,
    pocket_center: Tuple[float, float, float],
    crop_radius: float = POCKET_CROP_RADIUS,
) -> Tuple[int, int, float]:
    """Estimate the search space reduction from cropping.
    
    Returns:
        (original_atoms, cropped_atoms, reduction_percent)
    """
    atoms = _parse_pdb_atoms(protein_pdb)
    original_count = len(atoms)
    
    effective_radius = crop_radius + POCKET_CROP_BUFFER
    near_residues = _get_residues_near_center(atoms, pocket_center, effective_radius)
    cropped_atoms = [a for a in atoms if (a['chain'], a['res_num']) in near_residues]
    cropped_count = len(cropped_atoms)
    
    reduction = 100.0 * (1.0 - cropped_count / original_count) if original_count > 0 else 0.0
    
    return original_count, cropped_count, reduction


print("✓ Protein cropping functions loaded")
print(f"  Default crop radius: {POCKET_CROP_RADIUS}Å (+ {POCKET_CROP_BUFFER}Å buffer)")
print(f"  Include full residues: {INCLUDE_FULL_RESIDUES}")

✓ Protein cropping functions loaded
  Default crop radius: 6.0Å (+ 3.0Å buffer)
  Include full residues: True


In [7]:
import sys
import dgl
import dgl.function as fn
from copy import deepcopy
from types import SimpleNamespace

import yaml
from rdkit.Geometry import Point3D

# ── DGL compatibility shim ────────────────────────────────────────────────
# Newer DGL versions removed dgl.function.copy_edge (renamed to copy_e).
# EquiBind's model code still uses copy_edge, so we patch it back in.
if not hasattr(fn, 'copy_edge'):
    fn.copy_edge = fn.copy_e
    print("  ✓ Patched dgl.function.copy_edge → copy_e (DGL compat shim)")

# ── 0. Import EquiBind in-process ─────────────────────────────────────────
#    Add EquiBind to sys.path so we can import its modules directly,
#    avoiding subprocess + conda activation overhead per call.

  ✓ Patched dgl.function.copy_edge → copy_e (DGL compat shim)


# Pocket-Guided Docking

In [8]:
# ============================================================================
# POCKET-GUIDED EQUIBIND DOCKING (Multi-Pose per Pocket)
# ============================================================================
# Runs EquiBind docking guided by fpocket and p2rank pocket predictions,
# plus unguided (natural) EquiBind poses. Then produces statistics on how
# many unguided poses overlap with fpocket / p2rank pockets.
#
# Generates N poses per pocket using different RDKit conformers, each
# translated to the pocket center as starting geometry.
#
# REFACTORED: EquiBind is now loaded IN-PROCESS (no subprocess/conda overhead).
#   - Model loaded once globally.
#   - Receptor graph loaded once per protein and cached.
#   - Each docking call is a direct Python function call (~100x faster).
#
# All runtime output is governed by the DockingMonitor (see Signal Monitor
# cell above). Set MONITOR_LEVEL in Config to control verbosity:
#   MonitorLevel.ALL      → full trace
#   MonitorLevel.WARNING  → rejected poses & failures only
#   MonitorLevel.CRITICAL → fatal errors only
# ============================================================================

import sys
import dgl
from copy import deepcopy
from types import SimpleNamespace

import yaml
from rdkit.Geometry import Point3D

# ── 0. Import EquiBind in-process ─────────────────────────────────────────
#    Add EquiBind to sys.path so we can import its modules directly,
#    avoiding subprocess + conda activation overhead per call.

_equibind_dir = str(EQUIBIND_DIR)
if _equibind_dir not in sys.path:
    sys.path.insert(0, _equibind_dir)

from models.equibind import EquiBind as EquiBindModel
from commons.process_mols import (
    get_receptor_inference,
    get_rec_graph,
    get_lig_graph_revised,
    get_geometry_graph,
)
from commons.geometry_utils import (
    rigid_transform_Kabsch_3D,
    get_torsions,
    get_dihedral_vonMises,
    apply_changes,
)
from commons.utils import seed_all


# ── 0a. Load EquiBind model ONCE ──────────────────────────────────────────

def _load_equibind_model(equibind_dir: Path, device: str = "cuda"):
    """Load EquiBind model and train args. Called once at startup."""
    ckpt_path = equibind_dir / "runs" / "flexible_self_docking" / "best_checkpoint.pt"
    train_args_path = ckpt_path.parent / "train_arguments.yaml"

    with open(train_args_path) as f:
        train_args = yaml.safe_load(f)

    # Critical: set noise_initial=0 for inference
    train_args["model_parameters"]["noise_initial"] = 0

    args = SimpleNamespace(**train_args)
    args.checkpoint = str(ckpt_path)
    args.use_rdkit_coords = args.dataset_params.get("use_rdkit_coords", True)
    args.device = device

    dev = torch.device("cuda:0" if torch.cuda.is_available() and device == "cuda" else "cpu")
    checkpoint = torch.load(str(ckpt_path), map_location=dev)

    model = EquiBindModel(
        device=dev,
        lig_input_edge_feats_dim=15,
        rec_input_edge_feats_dim=27,
        **args.model_parameters,
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(dev)
    model.eval()

    seed_all(args.seed)

    monitor.info(f"EquiBind model loaded from {ckpt_path}")
    monitor.info(f"  Device: {dev}")
    return model, args, dev


def _load_receptor_graph(protein_pdb: Path, args: SimpleNamespace):
    """Build the EquiBind receptor graph for a protein. Cached per protein."""
    dp = args.dataset_params
    rec, rec_coords, c_alpha_coords, n_coords, c_coords = get_receptor_inference(str(protein_pdb))
    rec_graph = get_rec_graph(
        rec, rec_coords, c_alpha_coords, n_coords, c_coords,
        use_rec_atoms=dp["use_rec_atoms"],
        rec_radius=dp["rec_graph_radius"],
        surface_max_neighbors=dp["surface_max_neighbors"],
        surface_graph_cutoff=dp["surface_graph_cutoff"],
        surface_mesh_cutoff=dp["surface_mesh_cutoff"],
        c_alpha_max_neighbors=dp["c_alpha_max_neighbors"],
    )
    return rec_graph


# Load model once now
_eb_model, _eb_args, _eb_device = _load_equibind_model(EQUIBIND_DIR, EQUIBIND_DEVICE)

# Cache: protein_path_str → rec_graph
_rec_graph_cache: Dict[str, object] = {}


# ── 0b. In-process single-ligand docking ──────────────────────────────────

def _run_corrections_inproc(lig, lig_coord, predicted_coords):
    """Post-process: torsion fitting + Kabsch alignment (identical to EquiBind's run_corrections)."""
    input_coords = lig_coord.detach().cpu()
    prediction = predicted_coords.detach().cpu()

    lig_input = deepcopy(lig)
    conf = lig_input.GetConformer()
    for i in range(lig_input.GetNumAtoms()):
        x, y, z = input_coords.numpy()[i]
        conf.SetAtomPosition(i, Point3D(float(x), float(y), float(z)))

    lig_equibind = deepcopy(lig)
    conf = lig_equibind.GetConformer()
    for i in range(lig_equibind.GetNumAtoms()):
        x, y, z = prediction.numpy()[i]
        conf.SetAtomPosition(i, Point3D(float(x), float(y), float(z)))

    coords_pred = lig_equibind.GetConformer().GetPositions()
    Z_pt_cloud = coords_pred
    rotable_bonds = get_torsions([lig_input])
    new_dihedrals = np.zeros(len(rotable_bonds))
    for idx_t, r in enumerate(rotable_bonds):
        new_dihedrals[idx_t] = get_dihedral_vonMises(lig_input, lig_input.GetConformer(), r, Z_pt_cloud)
    optimized_mol = apply_changes(lig_input, new_dihedrals, rotable_bonds)
    optimized_conf = optimized_mol.GetConformer()
    coords_pred_optimized = optimized_conf.GetPositions()
    R, t = rigid_transform_Kabsch_3D(coords_pred_optimized.T, coords_pred.T)
    coords_pred_optimized = (R @ coords_pred_optimized.T).T + t.squeeze()
    for i in range(optimized_mol.GetNumAtoms()):
        x, y, z = coords_pred_optimized[i]
        optimized_conf.SetAtomPosition(i, Point3D(float(x), float(y), float(z)))
    return optimized_mol


def _dock_single_ligand_inproc(
    mol,
    rec_graph,
    model,
    args: SimpleNamespace,
    device: torch.device,
) -> Optional[Chem.Mol]:
    """Dock a single RDKit Mol in-process. Returns optimized Mol or None."""
    dp = args.dataset_params
    name = mol.GetProp("_Name") if mol.HasProp("_Name") else "unnamed"

    try:
        lig_graph = get_lig_graph_revised(
            mol, name,
            max_neighbors=dp["lig_max_neighbors"],
            use_rdkit_coords=args.use_rdkit_coords,
            radius=dp["lig_graph_radius"],
        )
    except Exception as e:
        monitor.warning(f"Ligand graph build failed for {name}: {e}")
        return None

    # Ensure new_x is set (required by model forward)
    lig_graph.ndata["new_x"] = lig_graph.ndata["x"]

    geometry_graph = get_geometry_graph(mol) if dp.get("geometry_regularization", True) else None

    # Save input coords for post-processing
    lig_coord = lig_graph.ndata["new_x"].clone()

    # Move graphs to device
    lig_graph = lig_graph.to(device)
    rec_graph_dev = rec_graph.to(device)
    geom_graph_dev = geometry_graph.to(device) if geometry_graph is not None else None

    try:
        with torch.no_grad():
            predictions = model(lig_graph, rec_graph_dev, geom_graph_dev)
        predicted_coords = predictions[0][0]  # (n_atoms, 3)
    except Exception as e:
        monitor.warning(f"EquiBind forward failed for {name}: {e}")
        return None

    try:
        optimized_mol = _run_corrections_inproc(mol, lig_coord, predicted_coords)
        return optimized_mol
    except Exception as e:
        monitor.warning(f"Post-processing failed for {name}: {e}")
        # Fallback: return mol with raw predicted coords
        out_mol = deepcopy(mol)
        conf = out_mol.GetConformer()
        coords_np = predicted_coords.detach().cpu().numpy()
        for i in range(out_mol.GetNumAtoms()):
            conf.SetAtomPosition(i, Point3D(*coords_np[i].tolist()))
        return out_mol


# ── 1. Parse fpocket results ──────────────────────────────────────────────

@dataclass
class PocketInfo:
    """Generic pocket descriptor from fpocket or p2rank."""
    source: str            # "fpocket" or "p2rank"
    pocket_id: int         # rank / pocket number (original)
    unique_id: str         # globally unique ID (e.g. "fpocket_cleaned_p001")
    score: float
    center: Tuple[float, float, float]
    radius: float          # rough extent
    protein_name: str
    extra: dict = field(default_factory=dict)

    def to_dict(self):
        return {
            "source": self.source,
            "pocket_id": self.pocket_id,
            "unique_id": self.unique_id,
            "score": self.score,
            "center": list(self.center),
            "radius": self.radius,
            "protein_name": self.protein_name,
            "extra": self.extra,
        }


def _parse_fpocket_pocket_centroid(pocket_pdb: Path) -> Tuple[Tuple[float, float, float], float]:
    """Compute centroid and approximate radius from fpocket pocket_*_atm.pdb."""
    coords = []
    with open(pocket_pdb, "r") as fh:
        for line in fh:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                try:
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    coords.append((x, y, z))
                except (ValueError, IndexError):
                    continue
    if not coords:
        return (0.0, 0.0, 0.0), 0.0
    arr = np.array(coords)
    centroid = tuple(arr.mean(axis=0))
    dists = np.linalg.norm(arr - arr.mean(axis=0), axis=1)
    radius = float(dists.max())
    return centroid, radius


def parse_fpocket_results(fpocket_dir: Path, protein_name: str, n_top: int = N_TOP_POCKETS) -> List[PocketInfo]:
    """Parse top-n fpocket pockets for a given protein.
    
    Uses unique IDs that include the directory variant (e.g. 'cleaned' vs raw)
    to avoid collisions when multiple _out directories match.
    """
    out_dirs = sorted(fpocket_dir.glob(f"{protein_name}*_out"))
    pockets: List[PocketInfo] = []
    for out_dir in out_dirs:
        pockets_dir = out_dir / "pockets"
        if not pockets_dir.exists():
            continue

        # Derive a short variant tag from the directory name
        dir_suffix = out_dir.name[len(protein_name):]
        variant = dir_suffix.replace("_out", "").strip("_") or "raw"

        # Parse info file for scores
        info_file = list(out_dir.glob("*_info.txt"))
        scores: Dict[int, float] = {}
        if info_file:
            with open(info_file[0]) as fh:
                current_pocket = None
                for line in fh:
                    line_s = line.strip()
                    if line_s.startswith("Pocket ") and ":" in line_s:
                        try:
                            current_pocket = int(line_s.split()[1])
                        except (ValueError, IndexError):
                            current_pocket = None
                    elif current_pocket is not None and line_s.startswith("Score"):
                        try:
                            scores[current_pocket] = float(line_s.split(":")[-1].strip())
                        except ValueError:
                            pass

        # Iterate over pocket PDB files
        pocket_pdbs = sorted(pockets_dir.glob("pocket*_atm.pdb"))
        for ppdb in pocket_pdbs:
            try:
                pocket_num = int(ppdb.stem.replace("pocket", "").replace("_atm", ""))
            except ValueError:
                continue
            centroid, radius = _parse_fpocket_pocket_centroid(ppdb)
            unique_id = f"fpocket_{variant}_p{pocket_num:03d}"
            pockets.append(PocketInfo(
                source="fpocket",
                pocket_id=pocket_num,
                unique_id=unique_id,
                score=scores.get(pocket_num, 0.0),
                center=centroid,
                radius=radius,
                protein_name=protein_name,
                extra={"pdb_file": str(ppdb), "out_dir": str(out_dir), "variant": variant},
            ))

    pockets.sort(key=lambda p: p.score, reverse=True)
    return pockets[:n_top]


# ── 2. Parse p2rank results ──────────────────────────────────────────────

def parse_p2rank_results(p2rank_dir: Path, protein_name: str, n_top: int = N_TOP_POCKETS) -> List[PocketInfo]:
    """Parse top-n p2rank pockets for a given protein from predictions CSV."""
    pockets: List[PocketInfo] = []
    pred_files = sorted(p2rank_dir.glob(f"{protein_name}*_predictions.csv"))
    for pred_file in pred_files:
        fname_stem = pred_file.stem
        base = fname_stem.replace("_predictions", "")
        suffix_part = base[len(protein_name):].replace(".pdb", "").strip("_")
        variant = suffix_part or "raw"

        with open(pred_file) as fh:
            reader = csv.DictReader(fh)
            for row in reader:
                try:
                    def _g(key):
                        for k, v in row.items():
                            if k.strip() == key:
                                return v.strip()
                        return None

                    rank = int(_g("rank"))
                    score = float(_g("score"))
                    prob = float(_g("probability"))
                    cx = float(_g("center_x"))
                    cy = float(_g("center_y"))
                    cz = float(_g("center_z"))
                    name = (_g("name") or "").strip()
                    sas = float(_g("sas_points") or 0)
                except (TypeError, ValueError):
                    continue

                radius = max(5.0, np.sqrt(sas) * 0.5)
                unique_id = f"p2rank_{variant}_p{rank:03d}"
                pockets.append(PocketInfo(
                    source="p2rank",
                    pocket_id=rank,
                    unique_id=unique_id,
                    score=score,
                    center=(cx, cy, cz),
                    radius=radius,
                    protein_name=protein_name,
                    extra={
                        "probability": prob,
                        "sas_points": sas,
                        "pred_file": str(pred_file),
                        "name": name,
                        "variant": variant,
                    },
                ))

    pockets.sort(key=lambda p: p.score, reverse=True)
    return pockets[:n_top]


# ── 3. Pocket-guided EquiBind docking (IN-PROCESS) ───────────────────────

def _translate_sdf_to_pocket(sdf_path: Path, pocket_center: Tuple[float, float, float],
                              output_path: Path) -> bool:
    """Translate a ligand SDF so its centroid is at the pocket center."""
    try:
        suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False)
        mol = next(iter(suppl), None)
        if mol is None:
            mol = Chem.MolFromMol2File(str(sdf_path), removeHs=False)
        if mol is None:
            return False

        mol = Chem.RWMol(mol)
        conf = mol.GetConformer()
        positions = conf.GetPositions()
        centroid = positions.mean(axis=0)
        shift = np.array(pocket_center) - centroid

        for i in range(mol.GetNumAtoms()):
            pos = conf.GetAtomPosition(i)
            conf.SetAtomPosition(i, (pos.x + shift[0], pos.y + shift[1], pos.z + shift[2]))

        writer = Chem.SDWriter(str(output_path))
        writer.write(mol)
        writer.close()
        return output_path.exists() and output_path.stat().st_size > 0
    except Exception as e:
        monitor.warning(f"Could not translate ligand to pocket: {e}")
        return False


def _run_equibind(protein_pdb: Path, ligand_file: Path, output_dir: Path,
                  seed: int = 42, device: str = "cpu") -> Tuple[bool, Optional[Path], str]:
    """Run EquiBind IN-PROCESS (no subprocess) for a single ligand.

    Uses the globally loaded model and caches receptor graphs per protein.
    The model, args, and device are stored in module-level variables
    _eb_model, _eb_args, _eb_device set at cell startup.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    # ── Monitor: log the outgoing call ──
    monitor.equibind_call(
        protein=protein_pdb.name,
        ligand=ligand_file.name,
        output_dir=str(output_dir),
        seed=seed,
        device=device,
    )

    # ── Get or cache receptor graph ──
    protein_key = str(protein_pdb)
    if protein_key not in _rec_graph_cache:
        monitor.info(f"Building receptor graph for {protein_pdb.name} (cached for reuse)")
        try:
            _rec_graph_cache[protein_key] = _load_receptor_graph(protein_pdb, _eb_args)
        except Exception as e:
            err_msg = f"Receptor graph build failed: {e}"
            monitor.equibind_return(False, None, err_msg)
            return False, None, err_msg

    rec_graph = _rec_graph_cache[protein_key]

    # ── Load ligand mol ──
    try:
        suffix = ligand_file.suffix.lower()
        if suffix == ".sdf":
            suppl = Chem.SDMolSupplier(str(ligand_file), sanitize=False, removeHs=False)
            mol = next(iter(suppl), None)
        elif suffix == ".mol2":
            mol = Chem.MolFromMol2File(str(ligand_file), removeHs=False)
        elif suffix == ".pdb":
            mol = Chem.MolFromPDBFile(str(ligand_file), removeHs=False)
        else:
            mol = None

        if mol is None:
            err_msg = f"Could not load ligand from {ligand_file.name}"
            monitor.equibind_return(False, None, err_msg)
            return False, None, err_msg

        Chem.SanitizeMol(mol)
        if not mol.HasProp("_Name"):
            mol.SetProp("_Name", ligand_file.stem)
    except Exception as e:
        err_msg = f"Ligand loading failed: {e}"
        monitor.equibind_return(False, None, err_msg)
        return False, None, err_msg

    # ── Seed for this call ──
    seed_all(seed)

    # ── Run in-process docking ──
    docked_mol = _dock_single_ligand_inproc(mol, rec_graph, _eb_model, _eb_args, _eb_device)

    if docked_mol is None:
        err_msg = "EquiBind inference returned None"
        monitor.equibind_return(False, None, err_msg)
        return False, None, err_msg

    # ── Write output SDF ──
    output_sdf = output_dir / "output.sdf"
    try:
        writer = Chem.SDWriter(str(output_sdf))
        writer.write(docked_mol)
        writer.close()
    except Exception as e:
        err_msg = f"Failed to write output SDF: {e}"
        monitor.equibind_return(False, None, err_msg)
        return False, None, err_msg

    if output_sdf.exists() and output_sdf.stat().st_size > 0:
        monitor.equibind_return(True, str(output_sdf))
        return True, output_sdf, ""
    else:
        err_msg = "Output SDF empty or missing"
        monitor.equibind_return(False, None, err_msg)
        return False, None, err_msg


def _sdf_centroid(sdf_path: Path) -> Optional[Tuple[float, float, float]]:
    """Compute centroid from SDF file."""
    try:
        coords = []
        with open(sdf_path) as f:
            lines = f.readlines()
        if len(lines) < 5:
            return None
        parts = lines[3].split()
        n_atoms = int(parts[0])
        for i in range(4, min(4 + n_atoms, len(lines))):
            p = lines[i].split()
            if len(p) >= 3:
                coords.append([float(p[0]), float(p[1]), float(p[2])])
        if not coords:
            return None
        return tuple(np.array(coords).mean(axis=0))
    except Exception:
        return None


def _pocket_distance(a: Tuple[float, float, float], b: Tuple[float, float, float]) -> float:
    """Euclidean distance between two 3D points."""
    return float(np.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2))


def _generate_conformer_sdfs(ligand_file: Path, output_dir: Path,
                              seeds: list = None, per_seed: int = None) -> List[Tuple[Path, int]]:
    """Generate diverse RDKit conformers, return list of (sdf_path, seed)."""
    if seeds is None:
        seeds = RDKIT_SEEDS
    if per_seed is None:
        per_seed = CONFORMERS_PER_SEED

    output_dir.mkdir(parents=True, exist_ok=True)

    suffix = ligand_file.suffix.lower()
    try:
        if suffix == ".sdf":
            suppl = Chem.SDMolSupplier(str(ligand_file), removeHs=False)
            mol = next(iter(suppl), None)
        elif suffix == ".mol2":
            mol = Chem.MolFromMol2File(str(ligand_file), removeHs=False)
        elif suffix == ".pdb":
            mol = Chem.MolFromPDBFile(str(ligand_file), removeHs=False)
        else:
            mol = None
    except Exception:
        mol = None

    if mol is None:
        monitor.warning(f"Could not load molecule from {ligand_file.name}")
        return []

    mol = Chem.AddHs(mol)
    conformer_files: List[Tuple[Path, int]] = []

    for seed in seeds:
        params = AllChem.ETKDGv3()
        params.randomSeed = seed
        params.numThreads = 0
        params.useRandomCoords = True
        params.maxIterations = 500
        try:
            cids = AllChem.EmbedMultipleConfs(mol, numConfs=per_seed, params=params)
        except Exception:
            cids = []
        for ci, cid in enumerate(cids):
            try:
                AllChem.MMFFOptimizeMolecule(mol, confId=cid, maxIters=200)
            except Exception:
                pass
            cpath = output_dir / f"seed{seed}_c{ci}.sdf"
            try:
                w = Chem.SDWriter(str(cpath))
                w.write(mol, confId=cid)
                w.close()
                if cpath.exists() and cpath.stat().st_size > 0:
                    conformer_files.append((cpath, seed))
            except Exception:
                pass

    monitor.info(f"RDKit generated {len(conformer_files)} conformers for {ligand_file.name}")
    return conformer_files


@dataclass
class GuidedPoseResult:
    """One docked pose from pocket-guided or unguided EquiBind."""
    protein_name: str
    ligand_name: str
    mode: str              # "fpocket", "p2rank", "unguided"
    pocket_id: Optional[int]
    pocket_unique_id: Optional[str]
    pose_num: int          # pose number within this pocket (1..POSES_PER_POCKET)
    pocket_center: Optional[Tuple[float, float, float]]
    pose_centroid: Optional[Tuple[float, float, float]]
    sdf_path: Optional[Path]
    success: bool
    error: str = ""

    def to_dict(self):
        return {
            "protein_name": self.protein_name,
            "ligand_name": self.ligand_name,
            "mode": self.mode,
            "pocket_id": self.pocket_id,
            "pocket_unique_id": self.pocket_unique_id,
            "pose_num": self.pose_num,
            "pocket_center": list(self.pocket_center) if self.pocket_center else None,
            "pose_centroid": list(self.pose_centroid) if self.pose_centroid else None,
            "sdf_path": str(self.sdf_path) if self.sdf_path else None,
            "success": self.success,
            "error": self.error,
        }


def _ensure_ligand_sdf(ligand_file: Path, prep_dir: Path) -> Path:
    """Ensure ligand is available as SDF; convert if needed."""
    ligand_name = ligand_file.stem
    lig_sdf = prep_dir / f"{ligand_name}.sdf"
    if lig_sdf.exists():
        return lig_sdf

    suffix = ligand_file.suffix.lower()
    if suffix == ".sdf":
        shutil.copy2(ligand_file, lig_sdf)
    elif suffix == ".mol2":
        mol = Chem.MolFromMol2File(str(ligand_file), removeHs=False)
        if mol:
            mol = Chem.AddHs(mol)
            AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
            w = Chem.SDWriter(str(lig_sdf)); w.write(mol); w.close()
        else:
            shutil.copy2(ligand_file, lig_sdf)
    elif suffix == ".pdb":
        mol = Chem.MolFromPDBFile(str(ligand_file), removeHs=False)
        if mol:
            mol = Chem.AddHs(mol)
            AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
            w = Chem.SDWriter(str(lig_sdf)); w.write(mol); w.close()
        else:
            shutil.copy2(ligand_file, lig_sdf)
    monitor.info(f"Prepared ligand SDF: {lig_sdf.name}")
    return lig_sdf


def _dock_pocket_multi_pose(
    pocket: PocketInfo,
    prepared_protein: Path,
    lig_sdf: Path,
    conformer_files: List[Tuple[Path, int]],
    combo_dir: Path,
    prep_dir: Path,
    protein_name: str,
    ligand_name: str,
    poses_per_pocket: int,
    skip_existing: bool,
    force_pocket: bool = FORCE_POCKET,
    max_pocket_trials: int = MAX_POCKET_TRIALS,
    pocket_threshold: float = POCKET_MATCH_THRESHOLD,
    use_cropping: bool = USE_PROTEIN_CROPPING,
) -> List[GuidedPoseResult]:
    """Dock N poses into a single pocket using different conformers.

    If use_cropping is True, the protein is cropped to only include residues
    near the pocket center, physically constraining EquiBind's search space.
    
    If force_pocket is True, each docked pose is checked: if the resulting
    centroid is farther than pocket_threshold from the pocket center, the
    pose is rejected and the next conformer is tried, up to
    max_pocket_trials attempts per desired pose.
    """
    mode = pocket.source  # "fpocket" or "p2rank"
    uid = pocket.unique_id
    results: List[GuidedPoseResult] = []
    
    # ── Protein cropping for pocket-constrained docking ──
    cropped_protein = None
    crop_offset = np.zeros(3)
    if use_cropping:
        cropped_protein, crop_offset = get_cropped_protein_for_pocket(
            protein_pdb=prepared_protein,
            pocket=pocket,
            prep_dir=prep_dir,
            crop_radius=POCKET_CROP_RADIUS,
        )
        if cropped_protein is not None:
            # Build receptor graph for cropped protein (cached separately)
            cropped_key = str(cropped_protein)
            if cropped_key not in _rec_graph_cache:
                monitor.info(f"Building receptor graph for cropped protein {cropped_protein.name}")
                _rec_graph_cache[cropped_key] = _load_receptor_graph(cropped_protein, _eb_args)
            monitor.info(f"Using cropped protein for {uid} (pocket at origin)")
        else:
            monitor.warning(f"Protein cropping failed for {uid}, falling back to full protein")

    # Check how many poses already exist for this pocket
    existing_count = 0
    for pose_num in range(1, poses_per_pocket + 1):
        final_sdf = combo_dir / f"{uid}_pose{pose_num:02d}.sdf"
        if final_sdf.exists() and skip_existing:
            centroid = _sdf_centroid(final_sdf)
            results.append(GuidedPoseResult(
                protein_name=protein_name, ligand_name=ligand_name,
                mode=mode, pocket_id=pocket.pocket_id,
                pocket_unique_id=uid, pose_num=pose_num,
                pocket_center=pocket.center, pose_centroid=centroid,
                sdf_path=final_sdf, success=True,
            ))
            existing_count += 1

    if existing_count >= poses_per_pocket:
        monitor.info(f"{uid}: all {poses_per_pocket} poses already cached, skipping")
        return results

    # Generate poses using conformers translated to pocket center
    poses_done = existing_count
    conf_idx = 0

    while poses_done < poses_per_pocket and conf_idx < len(conformer_files):
        pose_num = poses_done + 1
        final_sdf = combo_dir / f"{uid}_pose{pose_num:02d}.sdf"
        if final_sdf.exists() and skip_existing:
            conf_idx += 1
            continue

        # Try up to max_pocket_trials conformers for this pose slot
        trials_for_this_pose = 0
        accepted = False

        while trials_for_this_pose < max_pocket_trials and conf_idx < len(conformer_files):
            conf_path, seed = conformer_files[conf_idx]
            conf_idx += 1
            trials_for_this_pose += 1

            # Translate this conformer to the target location:
            # - If using cropped protein: translate to ORIGIN (pocket is at origin in cropped frame)
            # - Otherwise: translate to pocket center in original frame
            translated = prep_dir / f"lig_{uid}_pose{pose_num:02d}_t{trials_for_this_pose:02d}.sdf"
            if cropped_protein is not None:
                # Cropped protein has pocket at origin, so translate ligand to origin
                target_center = (0.0, 0.0, 0.0)
            else:
                target_center = pocket.center
                
            ok = _translate_sdf_to_pocket(conf_path, target_center, translated)
            if not ok:
                ok = _translate_sdf_to_pocket(lig_sdf, target_center, translated)
            if not ok:
                monitor.warning(f"{uid} pose {pose_num} trial {trials_for_this_pose}: "
                                f"ligand translation failed, skipping conformer")
                continue

            # Use cropped protein if available, otherwise full protein
            dock_protein = cropped_protein if cropped_protein is not None else prepared_protein
            
            pose_dir = combo_dir / f"{uid}_run{pose_num:02d}_t{trials_for_this_pose:02d}"
            success, out_sdf, err = _run_equibind(dock_protein, translated, pose_dir,
                                                   seed=seed, device=EQUIBIND_DEVICE)

            if not success or out_sdf is None:
                # Clean up failed run directory
                try:
                    shutil.rmtree(pose_dir, ignore_errors=True)
                except Exception:
                    pass
                monitor.warning(f"{uid} pose {pose_num} trial {trials_for_this_pose}: DOCK FAIL")
                continue

            # If using cropped protein, translate pose back to original coordinate frame
            if cropped_protein is not None and np.any(crop_offset != 0):
                restored_sdf = pose_dir / "output_restored.sdf"
                if translate_pose_back_to_original_frame(out_sdf, crop_offset, restored_sdf):
                    out_sdf = restored_sdf
                else:
                    monitor.warning(f"{uid} pose {pose_num}: failed to restore coordinates")

            # Compute centroid of the docked pose (now in original frame)
            centroid = _sdf_centroid(out_sdf)

            # ── Pocket enforcement with optional clamping ──
            if force_pocket and centroid is not None:
                dist = _pocket_distance(centroid, pocket.center)
                if dist > pocket_threshold:
                    pose_label = f"{uid}_pose{pose_num:02d}"
                    
                    # Option 1: Clamp pose back to pocket (if enabled)
                    if CLAMP_POSE_TO_POCKET:
                        clamped_sdf = pose_dir / "output_clamped.sdf"
                        clamp_ok, orig_dist, new_dist = clamp_pose_to_pocket(
                            sdf_path=out_sdf,
                            pocket_center=pocket.center,
                            max_distance=pocket_threshold,
                            output_path=clamped_sdf,
                        )
                        if clamp_ok:
                            out_sdf = clamped_sdf
                            centroid = _sdf_centroid(out_sdf)
                            monitor.info(f"[CLAMP] {pose_label} clamped from {orig_dist:.1f}Å → {new_dist:.1f}Å")
                        else:
                            monitor.warning(f"{pose_label}: clamping failed, rejecting pose")
                            try:
                                shutil.rmtree(pose_dir, ignore_errors=True)
                            except Exception:
                                pass
                            continue  # try next conformer
                    else:
                        # Option 2: Reject pose (original behavior)
                        monitor.pose_rejected_from_pocket(
                            pose_id=pose_label, pocket_id=uid,
                            distance=dist, threshold=pocket_threshold,
                        )
                        # Clean up rejected pose directory
                        try:
                            shutil.rmtree(pose_dir, ignore_errors=True)
                        except Exception:
                            pass
                        continue  # try next conformer

            # Pose accepted
            shutil.copy2(out_sdf, final_sdf)
            # Clean up run directory after successful copy
            try:
                shutil.rmtree(pose_dir, ignore_errors=True)
            except Exception:
                pass
            centroid = _sdf_centroid(final_sdf)  # re-read from final location
            results.append(GuidedPoseResult(
                protein_name=protein_name, ligand_name=ligand_name,
                mode=mode, pocket_id=pocket.pocket_id,
                pocket_unique_id=uid, pose_num=pose_num,
                pocket_center=pocket.center, pose_centroid=centroid,
                sdf_path=final_sdf, success=True,
            ))
            poses_done += 1
            if force_pocket and centroid:
                dist = _pocket_distance(centroid, pocket.center)
                pose_label = f"{uid}_pose{pose_num:02d}"
                monitor.pose_accepted_in_pocket(
                    pose_id=pose_label, pocket_id=uid,
                    distance=dist, threshold=pocket_threshold,
                )
                monitor.info(f"{uid} pose {pose_num}/{poses_per_pocket}: OK "
                             f"(trial {trials_for_this_pose}/{max_pocket_trials})")
            else:
                monitor.info(f"{uid} pose {pose_num}/{poses_per_pocket}: OK")
            accepted = True
            break  # move to next pose slot

        if not accepted:
            error_msg = (f"No pose stayed in pocket after {trials_for_this_pose} trials"
                         if force_pocket else "All conformers exhausted")
            results.append(GuidedPoseResult(
                protein_name=protein_name, ligand_name=ligand_name,
                mode=mode, pocket_id=pocket.pocket_id,
                pocket_unique_id=uid, pose_num=pose_num,
                pocket_center=pocket.center, pose_centroid=None,
                sdf_path=None, success=False,
                error=error_msg,
            ))
            monitor.critical(f"{uid} pose {pose_num}/{poses_per_pocket}: EXHAUSTED "
                             f"({trials_for_this_pose} trials, none in pocket)")

    return results


def dock_guided_by_pockets(
    protein_pdb: Path,
    ligand_file: Path,
    fpocket_pockets: List[PocketInfo],
    p2rank_pockets: List[PocketInfo],
    poses_per_pocket: int = POSES_PER_POCKET,
    n_unguided: int = N_UNGUIDED_POSES,
    skip_existing: bool = SKIP_EXISTING,
    force_pocket: bool = FORCE_POCKET,
    max_pocket_trials: int = MAX_POCKET_TRIALS,
) -> List[GuidedPoseResult]:
    """
    Dock one protein-ligand pair:
      1. For each fpocket pocket: generate N poses (different conformers)
      2. For each p2rank pocket: same
      3. Generate n_unguided natural poses (different conformers, no translation)

    If force_pocket=True, docked poses whose centroid drifts beyond
    POCKET_MATCH_THRESHOLD from the pocket center are rejected, and
    up to max_pocket_trials conformers are tried per pose slot.

    Returns list of GuidedPoseResult.
    """
    protein_name = protein_pdb.stem
    ligand_name = ligand_file.stem
    combo = f"{ligand_name}__{protein_name}"

    combo_dir = POCKET_GUIDED_OUTPUT_DIR / combo
    combo_dir.mkdir(parents=True, exist_ok=True)

    results_file = combo_dir / "guided_results.json"

    # Check for cached results
    if skip_existing and results_file.exists():
        try:
            cached = json.loads(results_file.read_text())
            expected_total = (len(fpocket_pockets) + len(p2rank_pockets)) * poses_per_pocket + n_unguided
            cached_results = []
            for r in cached:
                cached_results.append(GuidedPoseResult(
                    protein_name=r["protein_name"],
                    ligand_name=r["ligand_name"],
                    mode=r["mode"],
                    pocket_id=r.get("pocket_id"),
                    pocket_unique_id=r.get("pocket_unique_id"),
                    pose_num=r.get("pose_num", 1),
                    pocket_center=tuple(r["pocket_center"]) if r.get("pocket_center") else None,
                    pose_centroid=tuple(r["pose_centroid"]) if r.get("pose_centroid") else None,
                    sdf_path=Path(r["sdf_path"]) if r.get("sdf_path") else None,
                    success=r["success"],
                    error=r.get("error", ""),
                ))
            if len(cached_results) >= expected_total:
                monitor.info(f"[SKIP] {combo}: loaded {len(cached_results)} cached results")
                return cached_results
        except Exception:
            pass

    # Prepare protein (copy + reduce)
    prep_dir = combo_dir / "prep"
    prep_dir.mkdir(parents=True, exist_ok=True)
    prepared_protein = prep_dir / f"{protein_name}_protein.pdb"
    if not prepared_protein.exists():
        reduce_exe = shutil.which("reduce")
        if reduce_exe:
            try:
                res = subprocess.run([reduce_exe, "-Quiet", str(protein_pdb)],
                                     capture_output=True, text=True, timeout=60)
                if res.returncode == 0 and res.stdout.strip():
                    prepared_protein.write_text(res.stdout)
                    monitor.info(f"Protein prepared with reduce: {prepared_protein.name}")
                else:
                    shutil.copy2(protein_pdb, prepared_protein)
                    monitor.info(f"Protein copied (reduce returned empty): {prepared_protein.name}")
            except Exception:
                shutil.copy2(protein_pdb, prepared_protein)
        else:
            shutil.copy2(protein_pdb, prepared_protein)
            monitor.info(f"Protein copied (reduce not available): {prepared_protein.name}")

    # Pre-cache the receptor graph for this protein (done once, reused for all pockets/ligands)
    protein_key = str(prepared_protein)
    if protein_key not in _rec_graph_cache:
        monitor.info(f"Pre-caching receptor graph for {prepared_protein.name} ...")
        _rec_graph_cache[protein_key] = _load_receptor_graph(prepared_protein, _eb_args)
        monitor.info(f"Receptor graph cached for {prepared_protein.name}")

    # Ensure ligand SDF
    lig_sdf = _ensure_ligand_sdf(ligand_file, prep_dir)

    # Generate conformers (shared across all pockets for this combo)
    conf_dir = prep_dir / "conformers"
    conformer_files = _generate_conformer_sdfs(ligand_file, conf_dir, RDKIT_SEEDS, CONFORMERS_PER_SEED)
    if not conformer_files:
        conformer_files = [(lig_sdf, 42)]
        monitor.warning(f"No RDKit conformers generated — using original ligand SDF as fallback")
    monitor.info(f"Using {len(conformer_files)} conformers for multi-pose docking")
    if force_pocket:
        monitor.info(f"Pocket enforcement ON: max {max_pocket_trials} trials/pose, "
                     f"threshold {POCKET_MATCH_THRESHOLD}Å")
    if USE_PROTEIN_CROPPING:
        monitor.info(f"Protein cropping ON: {POCKET_CROP_RADIUS}Å radius + {POCKET_CROP_BUFFER}Å buffer")

    all_results: List[GuidedPoseResult] = []

    # ── A. Guided by fpocket pockets ──
    monitor.section(f"fpocket-guided  ({len(fpocket_pockets)} pockets × {poses_per_pocket} poses "
                    f"= {len(fpocket_pockets) * poses_per_pocket} total)")
    for pocket in fpocket_pockets:
        monitor.info(f"Pocket {pocket.unique_id}  score={pocket.score:.2f}  "
                     f"center=({pocket.center[0]:.1f}, {pocket.center[1]:.1f}, {pocket.center[2]:.1f})  "
                     f"radius={pocket.radius:.1f}Å")
        pocket_results = _dock_pocket_multi_pose(
            pocket=pocket,
            prepared_protein=prepared_protein,
            lig_sdf=lig_sdf,
            conformer_files=conformer_files,
            combo_dir=combo_dir,
            prep_dir=prep_dir,
            protein_name=protein_name,
            ligand_name=ligand_name,
            poses_per_pocket=poses_per_pocket,
            skip_existing=skip_existing,
            force_pocket=force_pocket,
            max_pocket_trials=max_pocket_trials,
        )
        ok_count = sum(1 for r in pocket_results if r.success)
        fail_count = sum(1 for r in pocket_results if not r.success)
        monitor.info(f"  → {ok_count}/{poses_per_pocket} poses OK, {fail_count} failed")
        if fail_count > 0:
            monitor.warning(f"  {pocket.unique_id}: {fail_count} pose(s) could not be placed in pocket")
        all_results.extend(pocket_results)

    # ── B. Guided by p2rank pockets ──
    monitor.section(f"p2rank-guided  ({len(p2rank_pockets)} pockets × {poses_per_pocket} poses "
                    f"= {len(p2rank_pockets) * poses_per_pocket} total)")
    for pocket in p2rank_pockets:
        monitor.info(f"Pocket {pocket.unique_id}  score={pocket.score:.2f}  "
                     f"center=({pocket.center[0]:.1f}, {pocket.center[1]:.1f}, {pocket.center[2]:.1f})  "
                     f"radius={pocket.radius:.1f}Å")
        pocket_results = _dock_pocket_multi_pose(
            pocket=pocket,
            prepared_protein=prepared_protein,
            lig_sdf=lig_sdf,
            conformer_files=conformer_files,
            combo_dir=combo_dir,
            prep_dir=prep_dir,
            protein_name=protein_name,
            ligand_name=ligand_name,
            poses_per_pocket=poses_per_pocket,
            skip_existing=skip_existing,
            force_pocket=force_pocket,
            max_pocket_trials=max_pocket_trials,
        )
        ok_count = sum(1 for r in pocket_results if r.success)
        fail_count = sum(1 for r in pocket_results if not r.success)
        monitor.info(f"  → {ok_count}/{poses_per_pocket} poses OK, {fail_count} failed")
        if fail_count > 0:
            monitor.warning(f"  {pocket.unique_id}: {fail_count} pose(s) could not be placed in pocket")
        all_results.extend(pocket_results)

    # ── C. Unguided (natural) EquiBind poses ──
    monitor.section(f"Unguided EquiBind poses  ({n_unguided} requested)")

    unguided_count = 0
    for conf_path, seed in conformer_files:
        if unguided_count >= n_unguided:
            break

        pose_num = unguided_count + 1
        final_sdf = combo_dir / f"unguided_{pose_num:03d}.sdf"

        if final_sdf.exists() and skip_existing:
            centroid = _sdf_centroid(final_sdf)
            all_results.append(GuidedPoseResult(
                protein_name=protein_name, ligand_name=ligand_name,
                mode="unguided", pocket_id=None,
                pocket_unique_id=None, pose_num=pose_num,
                pocket_center=None, pose_centroid=centroid,
                sdf_path=final_sdf, success=True,
            ))
            unguided_count += 1
            monitor.info(f"Unguided pose {pose_num}/{n_unguided}: cached")
            continue

        pose_dir = combo_dir / f"unguided_run_{pose_num:03d}"
        success, out_sdf, err = _run_equibind(prepared_protein, conf_path, pose_dir,
                                               seed=seed, device=EQUIBIND_DEVICE)
        if success and out_sdf:
            shutil.copy2(out_sdf, final_sdf)
            centroid = _sdf_centroid(final_sdf)
            all_results.append(GuidedPoseResult(
                protein_name=protein_name, ligand_name=ligand_name,
                mode="unguided", pocket_id=None,
                pocket_unique_id=None, pose_num=pose_num,
                pocket_center=None, pose_centroid=centroid,
                sdf_path=final_sdf, success=True,
            ))
            unguided_count += 1
            monitor.info(f"Unguided pose {pose_num}/{n_unguided}: OK  "
                         f"centroid=({centroid[0]:.1f}, {centroid[1]:.1f}, {centroid[2]:.1f})"
                         if centroid else f"Unguided pose {pose_num}/{n_unguided}: OK")
        else:
            all_results.append(GuidedPoseResult(
                protein_name=protein_name, ligand_name=ligand_name,
                mode="unguided", pocket_id=None,
                pocket_unique_id=None, pose_num=pose_num,
                pocket_center=None, pose_centroid=None,
                sdf_path=None, success=False, error=err[:200],
            ))
            monitor.warning(f"Unguided pose {pose_num}/{n_unguided}: FAIL  error={err[:120]}")

        try:
            shutil.rmtree(pose_dir, ignore_errors=True)
        except Exception:
            pass

    # ── D. Check which unguided poses fall inside known pockets ──
    monitor.section("Unguided pose ↔ pocket proximity check")
    all_fp_pockets = fpocket_pockets
    all_pr_pockets = p2rank_pockets
    all_known_pockets = all_fp_pockets + all_pr_pockets

    for r in all_results:
        if r.mode != "unguided" or not r.success or r.pose_centroid is None:
            continue
        pose_label = f"unguided_{r.pose_num:03d} ({ligand_name})"
        in_any = False
        nearest_id = None
        nearest_dist = float("inf")

        for pocket in all_known_pockets:
            dist = _pocket_distance(r.pose_centroid, pocket.center)
            if dist < nearest_dist:
                nearest_dist = dist
                nearest_id = pocket.unique_id
            if dist <= POCKET_MATCH_THRESHOLD:
                in_any = True
                monitor.pose_inside_pocket(
                    pose_id=pose_label, pocket_id=pocket.unique_id,
                    distance=dist, threshold=POCKET_MATCH_THRESHOLD,
                    pocket_source=pocket.source,
                )

        if not in_any and nearest_id is not None:
            monitor.pose_outside_all_pockets(
                pose_id=pose_label, nearest_pocket=nearest_id,
                nearest_dist=nearest_dist, threshold=POCKET_MATCH_THRESHOLD,
            )

    # Summary for this combo
    ok_fp = sum(1 for r in all_results if r.mode == "fpocket" and r.success)
    ok_pr = sum(1 for r in all_results if r.mode == "p2rank" and r.success)
    ok_ug = sum(1 for r in all_results if r.mode == "unguided" and r.success)
    fail_total = sum(1 for r in all_results if not r.success)
    monitor.header(f"COMBO DONE  {combo}")
    print(f"  fpocket-guided: {ok_fp}  |  p2rank-guided: {ok_pr}  |  "
          f"unguided: {ok_ug}  |  failed: {fail_total}  |  "
          f"total OK: {ok_fp + ok_pr + ok_ug}")
    if fail_total > 0:
        monitor.warning(f"{combo}: {fail_total} pose(s) failed across all modes")

    # Save results
    with open(results_file, "w") as f:
        json.dump([r.to_dict() for r in all_results], f, indent=2)

    return all_results


# ── 4. Batch run across all protein-ligand pairs ─────────────────────────

def _match_protein_name_to_pockets(protein_pdb: Path) -> str:
    return protein_pdb.stem


def collect_input_files(directory: Path, extensions: List[str]) -> List[Path]:
    files = []
    for ext in extensions:
        files.extend(sorted(p for p in directory.glob(f"*{ext}") if p.is_file()))
    return sorted(set(files))


# Collect inputs — use only .pdb ligands to avoid duplicate docking of the same molecule
ligand_files = collect_input_files(drugs_dir, [".pdb"])
receptor_files = collect_input_files(receptors_dir, [".pdb"])

n_fp_total = N_TOP_POCKETS * POSES_PER_POCKET
n_pr_total = N_TOP_POCKETS * POSES_PER_POCKET
n_per_combo = n_fp_total + n_pr_total + N_UNGUIDED_POSES
total_combos = len(receptor_files) * len(ligand_files)

monitor.header("POCKET-GUIDED EQUIBIND DOCKING (Multi-Pose per Pocket)")
print(f"  Monitor level: {MONITOR_LEVEL.name}")
print(f"  EquiBind mode: IN-PROCESS (no subprocess overhead)")
print(f"  Proteins: {len(receptor_files)}")
for p in receptor_files:
    print(f"    - {p.name}")
print(f"  Ligands: {len(ligand_files)}")
for l in ligand_files:
    print(f"    - {l.name}")
print(f"\n  Pockets configuration:")
print(f"    Top fpocket pockets per protein: {N_TOP_POCKETS}")
print(f"    Top p2rank pockets per protein:  {N_TOP_POCKETS}")
print(f"    Poses per pocket:                {POSES_PER_POCKET}")
print(f"    fpocket poses per combo:         {n_fp_total}  ({N_TOP_POCKETS} × {POSES_PER_POCKET})")
print(f"    p2rank poses per combo:          {n_pr_total}  ({N_TOP_POCKETS} × {POSES_PER_POCKET})")
print(f"    Unguided poses per combo:        {N_UNGUIDED_POSES}")
print(f"    TOTAL poses per combo:           {n_per_combo}")
print(f"    Total combinations:              {total_combos}")
print(f"    Expected grand total poses:      {total_combos * n_per_combo}")
print(f"\n  Pocket enforcement:                {'ON' if FORCE_POCKET else 'OFF'}")
print(f"    Max trials per pose:             {MAX_POCKET_TRIALS}")
print(f"    Pocket match threshold:          {POCKET_MATCH_THRESHOLD} Å")
print(f"  Output directory:                  {POCKET_GUIDED_OUTPUT_DIR}")
print(f"  Logs directory:                    {LOG_DIR}")
print()

# Collect pockets for each protein
protein_fpocket: Dict[str, List[PocketInfo]] = {}
protein_p2rank: Dict[str, List[PocketInfo]] = {}

for prot in receptor_files:
    pname = _match_protein_name_to_pockets(prot)
    fp_pockets = parse_fpocket_results(fpocket_results_folder, pname, N_TOP_POCKETS)
    pr_pockets = parse_p2rank_results(p2rank_folder, pname, N_TOP_POCKETS)
    protein_fpocket[pname] = fp_pockets
    protein_p2rank[pname] = pr_pockets
    monitor.info(f"{pname}: fpocket={len(fp_pockets)} pockets {[p.unique_id for p in fp_pockets]}")
    monitor.info(f"{pname}: p2rank ={len(pr_pockets)} pockets {[p.unique_id for p in pr_pockets]}")

print()

# Run docking
all_guided_results: List[GuidedPoseResult] = []
start_time = time.time()

for idx, (protein, ligand) in enumerate(itertools.product(receptor_files, ligand_files), 1):
    pname = _match_protein_name_to_pockets(protein)
    monitor.header(f"[{idx}/{total_combos}] {protein.name} × {ligand.name}")

    fp_pockets = protein_fpocket.get(pname, [])
    pr_pockets = protein_p2rank.get(pname, [])

    if not fp_pockets and not pr_pockets:
        monitor.warning(f"No pockets found for protein {pname} — only unguided poses will be generated")

    results = dock_guided_by_pockets(
        protein_pdb=protein,
        ligand_file=ligand,
        fpocket_pockets=fp_pockets,
        p2rank_pockets=pr_pockets,
        poses_per_pocket=POSES_PER_POCKET,
        n_unguided=N_UNGUIDED_POSES,
        skip_existing=SKIP_EXISTING,
        force_pocket=FORCE_POCKET,
        max_pocket_trials=MAX_POCKET_TRIALS,
    )
    all_guided_results.extend(results)

elapsed = time.time() - start_time


# ── 5. Statistics: how many unguided poses fall in fpocket / p2rank pockets ──

def _distance(a, b):
    return np.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2)


monitor.header("STATISTICS: UNGUIDED POSES vs FPOCKET / P2RANK POCKETS")

# Build lookup of all pockets for each protein
all_pockets_by_protein: Dict[str, Dict[str, List[PocketInfo]]] = defaultdict(lambda: {"fpocket": [], "p2rank": []})
for pname, pockets in protein_fpocket.items():
    all_pockets_by_protein[pname]["fpocket"] = pockets
for pname, pockets in protein_p2rank.items():
    all_pockets_by_protein[pname]["p2rank"] = pockets

# Analyze unguided poses
unguided_results = [r for r in all_guided_results if r.mode == "unguided" and r.success and r.pose_centroid]
fpocket_guided = [r for r in all_guided_results if r.mode == "fpocket" and r.success]
p2rank_guided = [r for r in all_guided_results if r.mode == "p2rank" and r.success]

stats_rows = []
total_unguided = 0
total_in_fpocket = 0
total_in_p2rank = 0
total_in_either = 0
total_in_neither = 0

# Group unguided by protein
unguided_by_protein: Dict[str, List[GuidedPoseResult]] = defaultdict(list)
for r in unguided_results:
    unguided_by_protein[r.protein_name].append(r)

for pname, poses in sorted(unguided_by_protein.items()):
    fp_pockets = all_pockets_by_protein[pname]["fpocket"]
    pr_pockets = all_pockets_by_protein[pname]["p2rank"]

    for pose in poses:
        total_unguided += 1
        in_fp = False
        in_pr = False
        nearest_fp_dist = float("inf")
        nearest_pr_dist = float("inf")
        nearest_fp_id = None
        nearest_pr_id = None

        for pocket in fp_pockets:
            dist = _distance(pose.pose_centroid, pocket.center)
            if dist < nearest_fp_dist:
                nearest_fp_dist = dist
                nearest_fp_id = pocket.unique_id
            if dist <= POCKET_MATCH_THRESHOLD:
                in_fp = True

        for pocket in pr_pockets:
            dist = _distance(pose.pose_centroid, pocket.center)
            if dist < nearest_pr_dist:
                nearest_pr_dist = dist
                nearest_pr_id = pocket.unique_id
            if dist <= POCKET_MATCH_THRESHOLD:
                in_pr = True

        if in_fp:
            total_in_fpocket += 1
        if in_pr:
            total_in_p2rank += 1
        if in_fp or in_pr:
            total_in_either += 1
        if not in_fp and not in_pr:
            total_in_neither += 1

        stats_rows.append({
            "protein": pname,
            "ligand": pose.ligand_name,
            "pose_sdf": str(pose.sdf_path.name) if pose.sdf_path else "",
            "pose_centroid_x": round(pose.pose_centroid[0], 2),
            "pose_centroid_y": round(pose.pose_centroid[1], 2),
            "pose_centroid_z": round(pose.pose_centroid[2], 2),
            "in_fpocket": in_fp,
            "nearest_fpocket_id": nearest_fp_id,
            "nearest_fpocket_dist_A": round(nearest_fp_dist, 2) if nearest_fp_dist < float("inf") else None,
            "in_p2rank": in_pr,
            "nearest_p2rank_id": nearest_pr_id,
            "nearest_p2rank_dist_A": round(nearest_pr_dist, 2) if nearest_pr_dist < float("inf") else None,
            "in_any_pocket": in_fp or in_pr,
        })

import pandas as pd

stats_df = pd.DataFrame(stats_rows)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)

print(f"\n  Total unguided poses analyzed: {total_unguided}")
print(f"  Poses within an fpocket pocket (<{POCKET_MATCH_THRESHOLD}Å): {total_in_fpocket} ({100*total_in_fpocket/max(total_unguided,1):.1f}%)")
print(f"  Poses within a p2rank pocket  (<{POCKET_MATCH_THRESHOLD}Å): {total_in_p2rank} ({100*total_in_p2rank/max(total_unguided,1):.1f}%)")
print(f"  Poses in EITHER fpocket or p2rank:  {total_in_either} ({100*total_in_either/max(total_unguided,1):.1f}%)")
print(f"  Poses in NEITHER pocket:            {total_in_neither} ({100*total_in_neither/max(total_unguided,1):.1f}%)")

print(f"\n  Total guided docking runs:")
print(f"    fpocket-guided: {len(fpocket_guided)} successful poses")
print(f"    p2rank-guided:  {len(p2rank_guided)} successful poses")
print(f"    unguided:       {len(unguided_results)} successful poses")
print(f"\n  Total elapsed time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

# Per-protein summary
print("\n" + "-" * 80)
print("PER-PROTEIN SUMMARY")
print("-" * 80)
if not stats_df.empty:
    summary = stats_df.groupby("protein").agg(
        total_unguided=("in_any_pocket", "count"),
        in_fpocket=("in_fpocket", "sum"),
        in_p2rank=("in_p2rank", "sum"),
        in_any=("in_any_pocket", "sum"),
    ).reset_index()
    summary["pct_in_fpocket"] = (100 * summary["in_fpocket"] / summary["total_unguided"]).round(1)
    summary["pct_in_p2rank"] = (100 * summary["in_p2rank"] / summary["total_unguided"]).round(1)
    summary["pct_in_any"] = (100 * summary["in_any"] / summary["total_unguided"]).round(1)
    display(summary)

print("\nDetailed per-pose results:")
if not stats_df.empty:
    display(stats_df)

# Save stats
stats_csv = POCKET_GUIDED_OUTPUT_DIR / "unguided_vs_pockets_statistics.csv"
stats_df.to_csv(stats_csv, index=False)
print(f"\nStatistics saved to: {stats_csv}")

# Save full summary JSON
full_summary = {
    "timestamp": datetime.now().isoformat(),
    "config": {
        "n_top_pockets": N_TOP_POCKETS,
        "poses_per_pocket": POSES_PER_POCKET,
        "n_unguided_poses": N_UNGUIDED_POSES,
        "pocket_match_threshold_A": POCKET_MATCH_THRESHOLD,
        "force_pocket": FORCE_POCKET,
        "max_pocket_trials": MAX_POCKET_TRIALS,
        "monitor_level": MONITOR_LEVEL.name,
    },
    "totals": {
        "unguided_poses": total_unguided,
        "in_fpocket": total_in_fpocket,
        "in_p2rank": total_in_p2rank,
        "in_either": total_in_either,
        "in_neither": total_in_neither,
        "fpocket_guided_success": len(fpocket_guided),
        "p2rank_guided_success": len(p2rank_guided),
        "elapsed_s": round(elapsed, 1),
    },
    "monitor_counters": {
        "equibind_calls": monitor.call_count,
        "equibind_success": monitor.success_count,
        "equibind_fail": monitor.fail_count,
        "poses_in_pocket": monitor.in_pocket_count,
        "poses_rejected": monitor.rejected_count,
        "poses_outside_all": monitor.outside_pocket_count,
    },
    "all_results": [r.to_dict() for r in all_guided_results],
}
summary_json = POCKET_GUIDED_OUTPUT_DIR / "pocket_guided_summary.json"
with open(summary_json, "w") as f:
    json.dump(full_summary, f, indent=2)
print(f"\nFull summary saved to: {summary_json}")

# ── Print signal monitor summary ──
monitor.print_summary()

[2026-02-26 11:21:48.074893] [ Using Seed :  1  ]
    · EquiBind model loaded from /home/manndo/docking_tools/EquiBind/runs/flexible_self_docking/best_checkpoint.pt
    ·   Device: cuda:0

──────────────────────────────────────────────────────────────────────
  POCKET-GUIDED EQUIBIND DOCKING (Multi-Pose per Pocket)
──────────────────────────────────────────────────────────────────────
  Monitor level: ALL
  EquiBind mode: IN-PROCESS (no subprocess overhead)
  Proteins: 8
    - Orai1WT-MDSnap-Fr300.pdb
    - Orai1WT-MDSnap-Fr300_cleaned.pdb
    - Orai1WT-MDSnap-Fr400.pdb
    - Orai1WT-MDSnap-Fr400_cleaned.pdb
    - Orai1WT-MDSnap-Fr499.pdb
    - Orai1WT-MDSnap-Fr499_cleaned.pdb
    - Orai1WT-START-Fr0.pdb
    - Orai1WT-START-Fr0_cleaned.pdb
  Ligands: 5
    - 2abp-nh2-OPT.pdb
    - 2abp-nh3p-OPT.pdb
    - Synta-66-OPT-Singlet.pdb
    - gsk7975a-deprot-OPT.pdb
    - gsk7975a-prot-OPT.pdb

  Pockets configuration:
    Top fpocket pockets per protein: 5
    Top p2rank pockets per protein: 

,protein,total_unguided,in_fpocket,in_p2rank,in_any,pct_in_fpocket,pct_in_p2rank,pct_in_any
0,Orai1WT-MDSnap-Fr300,25,0,0,0,0.0,0.0,0.0
1,Orai1WT-MDSnap-Fr300_cleaned,25,0,0,0,0.0,0.0,0.0
2,Orai1WT-MDSnap-Fr400,25,4,0,4,16.0,0.0,16.0
3,Orai1WT-MDSnap-Fr400_cleaned,25,0,0,0,0.0,0.0,0.0
4,Orai1WT-MDSnap-Fr499,25,0,0,0,0.0,0.0,0.0
5,Orai1WT-MDSnap-Fr499_cleaned,25,0,0,0,0.0,0.0,0.0
6,Orai1WT-START-Fr0,25,0,0,0,0.0,0.0,0.0
7,Orai1WT-START-Fr0_cleaned,25,2,0,2,8.0,0.0,8.0



Detailed per-pose results:


,protein,ligand,pose_sdf,pose_centroid_x,pose_centroid_y,pose_centroid_z,in_fpocket,nearest_fpocket_id,nearest_fpocket_dist_A,in_p2rank,nearest_p2rank_id,nearest_p2rank_dist_A,in_any_pocket
0,Orai1WT-MDSnap-Fr300,2abp-nh2-OPT,unguided_001.sdf,-10.63,27.90,-5.16,False,fpocket_raw_p001,22.93,False,p2rank_raw_p001,26.58,False
1,Orai1WT-MDSnap-Fr300,2abp-nh2-OPT,unguided_002.sdf,-10.48,30.93,-2.56,False,fpocket_raw_p001,22.20,False,p2rank_raw_p002,25.31,False
2,Orai1WT-MDSnap-Fr300,2abp-nh2-OPT,unguided_003.sdf,-11.58,27.76,-3.86,False,fpocket_raw_p001,22.13,False,p2rank_raw_p001,26.55,False
3,Orai1WT-MDSnap-Fr300,2abp-nh2-OPT,unguided_004.sdf,-10.62,27.78,-7.06,False,fpocket_raw_p001,24.53,False,p2rank_raw_p001,27.06,False
4,Orai1WT-MDSnap-Fr300,2abp-nh2-OPT,unguided_005.sdf,-10.89,30.65,-2.69,False,fpocket_raw_p001,22.30,False,p2rank_raw_p002,25.62,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,Orai1WT-START-Fr0_cleaned,gsk7975a-prot-OPT,unguided_001.sdf,-7.42,21.78,-0.57,False,fpocket_raw_p005,11.44,False,p2rank_raw_p004,15.93,False
196,Orai1WT-START-Fr0_cleaned,gsk7975a-prot-OPT,unguided_002.sdf,-7.57,21.58,-0.63,False,fpocket_raw_p005,11.22,False,p2rank_raw_p004,16.11,False
197,Orai1WT-START-Fr0_cleaned,gsk7975a-prot-OPT,unguided_003.sdf,-7.57,21.79,-0.46,False,fpocket_raw_p005,11.45,False,p2rank_raw_p004,15.94,False
198,Orai1WT-START-Fr0_cleaned,gsk7975a-prot-OPT,unguided_004.sdf,-7.74,21.02,-0.91,False,fpocket_raw_p005,10.62,False,p2rank_raw_p004,16.55,False



Statistics saved to: /home/manndo/master_dev/equibind_pocket_guided/unguided_vs_pockets_statistics.csv

Full summary saved to: /home/manndo/master_dev/equibind_pocket_guided/pocket_guided_summary.json

══════════════════════════════════════════════════════════════════════
  SIGNAL MONITOR SUMMARY
══════════════════════════════════════════════════════════════════════
  EquiBind calls:        1400
  Successful returns:    1400
  Failed returns:        0
  Poses inside pocket:   1206
  Poses rejected/outside:194
    ├ guided rejected:   0
    └ unguided outside:  194
══════════════════════════════════════════════════════════════════════


In [9]:
de

NameError: name 'de' is not defined

# Exclusion Appraoch

In [ ]:
import hashlib
import itertools
import json
import os
import shutil
import subprocess
import time
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import numpy as np
from collections import defaultdict

# RDKit imports for conformer generation
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign

In [ ]:
de

In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these parameters as needed
# ============================================================================

# Paths
workspace_root = Path.cwd()
drugs_dir = workspace_root / "Drugs"
receptors_dir = workspace_root / "Orai"

# EquiBind paths
EQUIBIND_DIR = Path("/home/manndo/docking_tools/EquiBind")
EQUIBIND_MULTILIGAND_SCRIPT = EQUIBIND_DIR / "multiligand_inference.py"

# Number of unique poses to generate per protein-ligand combination
NUM_POSES: int = 30

# Number of RDKit conformers to generate (should be >= NUM_POSES, extras for diversity)
NUM_CONFORMERS: int = 35

# Maximum docking attempts per combination (fallback if conformers fail)
MAX_ATTEMPTS: int = NUM_POSES * 30

# Minimum RMSD (in Angstroms) between poses to consider them distinct
POSE_RMSD_THRESHOLD: float = 0.5

# RDKit conformer generation parameters
RDKIT_RANDOM_SEED: int = 42
RDKIT_NUM_THREADS: int = 0  # 0 = use all available threads
RDKIT_PRUNE_RMS_THRESH: float = 0.5  # Prune similar conformers during generation

# Device for EquiBind (cpu or cuda)
EQUIBIND_DEVICE = "cuda"

# Output directories
EQUIBIND_BATCH_DIR = workspace_root / "equibind_batches"
EQUIBIND_OUTPUT_DIR = workspace_root / "equibind_docked_poses"
EQUIBIND_BATCH_DIR.mkdir(parents=True, exist_ok=True)
EQUIBIND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check for reduce executable (for adding hydrogens to proteins)
REDUCE_EXECUTABLE = shutil.which("reduce")

# Overwrite settings
OVERWRITE_EXISTING = False  # Set to True to re-dock existing combinations

# Validate directories
if not drugs_dir.exists():
    raise FileNotFoundError(f"Ligand directory missing: {drugs_dir}")
if not receptors_dir.exists():
    raise FileNotFoundError(f"Receptor directory missing: {receptors_dir}")

print("=" * 80)
print("EquiBind Batch Docking Configuration")
print("=" * 80)
print(f"Number of poses per combination: {NUM_POSES}")
print(f"RDKit conformers to generate: {NUM_CONFORMERS}")
print(f"Maximum attempts per combination: {MAX_ATTEMPTS}")
print(f"Pose RMSD threshold: {POSE_RMSD_THRESHOLD} Å")
print(f"RDKit prune RMS threshold: {RDKIT_PRUNE_RMS_THRESH} Å")
print(f"EquiBind script: {EQUIBIND_MULTILIGAND_SCRIPT}")
print(f"EquiBind device: {EQUIBIND_DEVICE}")
print(f"Batch directory: {EQUIBIND_BATCH_DIR}")
print(f"Output directory: {EQUIBIND_OUTPUT_DIR}")
print(f"Reduce executable: {REDUCE_EXECUTABLE or 'NOT FOUND'}")
print()

# Validate EquiBind installation
if not EQUIBIND_MULTILIGAND_SCRIPT.exists():
    print(f"⚠️  WARNING: EquiBind multiligand_inference.py not found at {EQUIBIND_MULTILIGAND_SCRIPT}")
    print("   Please update EQUIBIND_DIR to point to your EquiBind installation")
else:
    print(f"✓ EquiBind script found")

EquiBind Batch Docking Configuration
Number of poses per combination: 30
RDKit conformers to generate: 35
Maximum attempts per combination: 900
Pose RMSD threshold: 0.5 Å
RDKit prune RMS threshold: 0.5 Å
EquiBind script: /home/manndo/docking_tools/EquiBind/multiligand_inference.py
EquiBind device: cuda
Batch directory: /home/manndo/master_dev/equibind_batches
Output directory: /home/manndo/master_dev/equibind_docked_poses
Reduce executable: NOT FOUND

✓ EquiBind script found


In [ ]:
@dataclass
class PoseInfo:
    """Information about a single docked pose."""
    pose_id: int
    sdf_path: Path
    centroid: Tuple[float, float, float]
    seed: int
    conformer_id: int
    
    def to_dict(self) -> dict:
        return {
            "pose_id": self.pose_id,
            "sdf_path": str(self.sdf_path),
            "centroid": list(self.centroid),
            "seed": self.seed,
            "conformer_id": self.conformer_id,
        }


@dataclass
class DockingResult:
    """Result of docking a protein-ligand combination."""
    protein_name: str
    ligand_name: str
    protein_path: Path
    ligand_path: Path
    output_dir: Path
    status: str  # "success", "partial", "failed", "skipped"
    poses: List[PoseInfo] = field(default_factory=list)
    num_attempts: int = 0
    num_conformers_generated: int = 0
    error_message: str = ""
    elapsed_time: float = 0.0
    
    def to_dict(self) -> dict:
        return {
            "protein_name": self.protein_name,
            "ligand_name": self.ligand_name,
            "protein_path": str(self.protein_path),
            "ligand_path": str(self.ligand_path),
            "output_dir": str(self.output_dir),
            "status": self.status,
            "poses": [p.to_dict() for p in self.poses],
            "num_poses": len(self.poses),
            "num_attempts": self.num_attempts,
            "num_conformers_generated": self.num_conformers_generated,
            "error_message": self.error_message,
            "elapsed_time": self.elapsed_time,
        }


def get_file_stem(path: Path) -> str:
    """Get clean filename stem without extension."""
    return path.stem.replace("_ligand", "").replace("_protein", "")


# ============================================================================
# RDKit Conformer Generation Functions
# ============================================================================

def generate_rdkit_conformers(
    ligand_path: Path,
    num_conformers: int = NUM_CONFORMERS,
    random_seed: int = RDKIT_RANDOM_SEED,
    prune_rms_thresh: float = RDKIT_PRUNE_RMS_THRESH,
) -> Tuple[Optional[Chem.Mol], List[int]]:
    """
    Generate multiple 3D conformers for a ligand using RDKit's distance geometry.
    
    Uses EmbedMultipleConfs with ETKDG (Experimental-Torsion basic Knowledge Distance Geometry)
    to generate diverse conformers.
    
    Args:
        ligand_path: Path to ligand file (SDF, MOL2, or PDB)
        num_conformers: Number of conformers to generate
        random_seed: Random seed for reproducibility
        prune_rms_thresh: Prune conformers with RMSD below this threshold
    
    Returns:
        Tuple of (RDKit Mol object with conformers, list of conformer IDs)
    """
    suffix = ligand_path.suffix.lower()
    
    # Load molecule based on file type
    try:
        if suffix == ".sdf":
            suppl = Chem.SDMolSupplier(str(ligand_path), removeHs=False)
            mol = next(iter(suppl), None)
        elif suffix == ".mol2":
            mol = Chem.MolFromMol2File(str(ligand_path), removeHs=False)
        elif suffix == ".pdb":
            mol = Chem.MolFromPDBFile(str(ligand_path), removeHs=False)
        else:
            print(f"  Warning: Unsupported file format {suffix}")
            return None, []
        
        if mol is None:
            print(f"  Warning: Could not load molecule from {ligand_path}")
            return None, []
        
        # Add hydrogens if not present
        mol = Chem.AddHs(mol)
        
        # Set up ETKDG parameters for conformer generation
        params = AllChem.ETKDGv3()
        params.randomSeed = random_seed
        params.numThreads = RDKIT_NUM_THREADS
        params.pruneRmsThresh = prune_rms_thresh
        params.useRandomCoords = True  # Better for difficult molecules
        
        # Generate conformers
        conf_ids = AllChem.EmbedMultipleConfs(
            mol, 
            numConfs=num_conformers,
            params=params
        )
        
        if len(conf_ids) == 0:
            # Fallback: try with random coordinates and less strict parameters
            print(f"  Retrying conformer generation with relaxed parameters...")
            params.useRandomCoords = True
            params.maxIterations = 500
            params.pruneRmsThresh = 0.1  # Less strict pruning
            conf_ids = AllChem.EmbedMultipleConfs(
                mol,
                numConfs=num_conformers,
                params=params
            )
        
        if len(conf_ids) == 0:
            print(f"  Warning: Could not generate conformers for {ligand_path.name}")
            return mol, []
        
        # Optimize conformers with MMFF force field
        results = AllChem.MMFFOptimizeMoleculeConfs(mol, numThreads=RDKIT_NUM_THREADS)
        
        # Filter out failed optimizations
        valid_conf_ids = [
            conf_id for conf_id, (converged, energy) in zip(conf_ids, results)
            if converged == 0  # 0 means optimization converged
        ]
        
        # If all failed, use original conformers
        if not valid_conf_ids:
            valid_conf_ids = list(conf_ids)
        
        return mol, valid_conf_ids
        
    except Exception as e:
        print(f"  Error generating conformers for {ligand_path.name}: {e}")
        return None, []


def save_conformer_to_sdf(mol: Chem.Mol, conf_id: int, output_path: Path) -> bool:
    """Save a specific conformer to an SDF file."""
    try:
        writer = Chem.SDWriter(str(output_path))
        writer.write(mol, confId=conf_id)
        writer.close()
        return output_path.exists() and output_path.stat().st_size > 0
    except Exception as e:
        print(f"  Error saving conformer to {output_path}: {e}")
        return False


def compute_centroid(sdf_path: Path) -> Optional[Tuple[float, float, float]]:
    """Compute the centroid of atom coordinates from an SDF file."""
    try:
        coords = []
        with open(sdf_path, 'r') as f:
            lines = f.readlines()
        
        if len(lines) < 5:
            return None
            
        counts_line = lines[3].strip()
        parts = counts_line.split()
        if len(parts) < 2:
            return None
        
        num_atoms = int(parts[0])
        
        for i in range(4, min(4 + num_atoms, len(lines))):
            parts = lines[i].split()
            if len(parts) >= 3:
                try:
                    x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
                    coords.append((x, y, z))
                except ValueError:
                    continue
        
        if not coords:
            return None
        
        coords_array = np.array(coords)
        centroid = coords_array.mean(axis=0)
        return tuple(centroid)
    except Exception as e:
        print(f"Warning: Could not compute centroid for {sdf_path}: {e}")
        return None


def compute_pose_rmsd(sdf1: Path, sdf2: Path) -> Optional[float]:
    """Compute RMSD between two poses from SDF files."""
    try:
        def read_coords(sdf_path: Path) -> Optional[np.ndarray]:
            coords = []
            with open(sdf_path, 'r') as f:
                lines = f.readlines()
            if len(lines) < 5:
                return None
            counts_line = lines[3].strip()
            parts = counts_line.split()
            if len(parts) < 2:
                return None
            num_atoms = int(parts[0])
            for i in range(4, min(4 + num_atoms, len(lines))):
                parts = lines[i].split()
                if len(parts) >= 3:
                    try:
                        x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
                        coords.append([x, y, z])
                    except ValueError:
                        continue
            return np.array(coords) if coords else None
        
        coords1 = read_coords(sdf1)
        coords2 = read_coords(sdf2)
        
        if coords1 is None or coords2 is None:
            return None
        if len(coords1) != len(coords2):
            return None
        
        diff = coords1 - coords2
        rmsd = np.sqrt((diff ** 2).sum() / len(coords1))
        return float(rmsd)
    except Exception as e:
        print(f"Warning: Could not compute RMSD: {e}")
        return None


def is_pose_unique(new_sdf: Path, existing_poses: List[PoseInfo], threshold: float) -> bool:
    """Check if a new pose is sufficiently different from existing poses."""
    if not existing_poses:
        return True
    
    for pose in existing_poses:
        rmsd = compute_pose_rmsd(new_sdf, pose.sdf_path)
        if rmsd is not None and rmsd < threshold:
            return False
    return True


def prepare_protein_with_reduce(protein_pdb: Path, output_dir: Path) -> Path:
    """Run reduce on protein to add hydrogens, or copy as-is if reduce unavailable."""
    output_pdb = output_dir / f"{protein_pdb.stem}_protein.pdb"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    if REDUCE_EXECUTABLE:
        try:
            cmd = [REDUCE_EXECUTABLE, "-Quiet", str(protein_pdb)]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            if result.returncode == 0 and result.stdout.strip():
                output_pdb.write_text(result.stdout)
                return output_pdb
        except Exception as e:
            print(f"  Warning: reduce failed for {protein_pdb.name}: {e}")
    
    # Fallback: copy protein as-is
    shutil.copy2(protein_pdb, output_pdb)
    return output_pdb


def prepare_ligand_file(ligand_path: Path, output_dir: Path) -> Path:
    """Prepare ligand file for EquiBind (copy SDF/MOL2/PDB with proper naming)."""
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = get_file_stem(ligand_path)
    output_path = output_dir / f"{stem}_ligand{ligand_path.suffix}"
    shutil.copy2(ligand_path, output_path)
    return output_path


def collect_files(root: Path, extensions: List[str]) -> List[Path]:
    """Return files with given extensions located directly inside root (no recursion)."""
    files = []
    for ext in extensions:
        files.extend(sorted(p for p in root.glob(f"*{ext}") if p.is_file()))
    return sorted(set(files))


print("Helper functions and RDKit conformer generation defined successfully.")

Helper functions and RDKit conformer generation defined successfully.


In [ ]:
# Collect input files
# Ligands: PDB, SDF, MOL2 files from drugs_dir
# Proteins: PDB files from receptors_dir

ligand_files = collect_files(drugs_dir, [".sdf"])
receptor_files = collect_files(receptors_dir, [".pdb"])

print("=" * 80)
print("INPUT FILES FOR EQUIBIND DOCKING")
print("=" * 80)

print(f"\nProteins ({len(receptor_files)} files from {receptors_dir}):")
for pdb in receptor_files:
    print(f"  - {pdb.name}")

print(f"\nLigands ({len(ligand_files)} files from {drugs_dir}):")
for lig in ligand_files:
    print(f"  - {lig.name}")

print(f"\nTotal docking combinations: {len(receptor_files) * len(ligand_files)}")
print(f"Poses per combination: {NUM_POSES}")
print(f"Expected total poses: {len(receptor_files) * len(ligand_files) * NUM_POSES}")

INPUT FILES FOR EQUIBIND DOCKING

Proteins (8 files from /home/manndo/master_dev/Orai):
  - Orai1WT-MDSnap-Fr300.pdb
  - Orai1WT-MDSnap-Fr300_cleaned.pdb
  - Orai1WT-MDSnap-Fr400.pdb
  - Orai1WT-MDSnap-Fr400_cleaned.pdb
  - Orai1WT-MDSnap-Fr499.pdb
  - Orai1WT-MDSnap-Fr499_cleaned.pdb
  - Orai1WT-START-Fr0.pdb
  - Orai1WT-START-Fr0_cleaned.pdb

Ligands (5 files from /home/manndo/master_dev/Drugs):
  - 2abp-nh2-OPT.sdf
  - 2abp-nh3p-OPT.sdf
  - Synta-66-OPT-Singlet.sdf
  - gsk7975a-deprot-OPT.sdf
  - gsk7975a-prot-OPT.sdf

Total docking combinations: 40
Poses per combination: 30
Expected total poses: 1200


# Spatial Exclusion Config

In [ ]:
# ============================================================================
# SPATIAL EXCLUSION DOCKING CONFIGURATION
# ============================================================================
from pathlib import Path
import shutil
import time
import json
from datetime import datetime
import itertools

# EquiBind paths (if not already defined)
EQUIBIND_DIR = Path("/home/manndo/docking_tools/EquiBind")
EQUIBIND_MULTILIGAND_SCRIPT = EQUIBIND_DIR / "multiligand_inference.py"

# Device for EquiBind (cpu or cuda)
EQUIBIND_DEVICE = "cuda"

# Output directories - SEPARATE folders for each method
workspace_root = Path.cwd()
EQUIBIND_BATCH_DIR = workspace_root / "equibind_batches"

# Method 1: Standard conformer-based docking (no exclusion)
CONFORMER_DOCKING_OUTPUT_DIR = workspace_root / "equibind_conformer_poses"

# Method 2: Spatial exclusion docking
SPATIAL_EXCLUSION_OUTPUT_DIR = workspace_root / "equibind_spatial_exclusion_poses"

# Legacy output directory (for backward compatibility)
EQUIBIND_OUTPUT_DIR = workspace_root / "equibind_docked_poses"

# Create all directories
EQUIBIND_BATCH_DIR.mkdir(parents=True, exist_ok=True)
CONFORMER_DOCKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPATIAL_EXCLUSION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EQUIBIND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# RDKit random seeds - use multiple seeds for maximum conformer diversity
# Each seed will generate a batch of conformers with different starting geometries
RDKIT_RANDOM_SEEDS: list = [42, 123, 456, 789, 1001, 2022, 3141, 5926, 8675, 9999]

# Number of conformers to generate per seed (total conformers = len(seeds) * CONFORMERS_PER_SEED)
CONFORMERS_PER_SEED: int = 20

# Number of distinct binding sites to discover per protein-ligand combination
NUM_BINDING_SITES: int = 10

# Exclusion radius (Angstroms) around each found binding site
# Poses with centroids within this distance of a previous site are rejected
EXCLUSION_RADIUS: float = 5.0  # Typical binding pocket is ~10-15Å across

# Minimum distance between binding site centroids to be considered distinct
MIN_SITE_DISTANCE: float = 3.0  # Å

# Maximum total conformers to try before giving up on finding a new site
MAX_CONFORMERS_PER_SITE: int = 90

print("=" * 80)
print("SPATIAL EXCLUSION DOCKING CONFIGURATION")
print("=" * 80)
print(f"EquiBind device: {EQUIBIND_DEVICE}")
print(f"\nOutput directories:")
print(f"  Conformer docking:     {CONFORMER_DOCKING_OUTPUT_DIR}")
print(f"  Spatial exclusion:     {SPATIAL_EXCLUSION_OUTPUT_DIR}")
print(f"\nRDKit random seeds: {RDKIT_RANDOM_SEEDS}")
print(f"Conformers per seed: {CONFORMERS_PER_SEED}")
print(f"Total conformers per batch: {len(RDKIT_RANDOM_SEEDS) * CONFORMERS_PER_SEED}")
print(f"\nSpatial exclusion settings:")
print(f"  Binding sites to discover: {NUM_BINDING_SITES}")
print(f"  Maximum conformers to try per site: {MAX_CONFORMERS_PER_SITE}")
print(f"  Exclusion radius: {EXCLUSION_RADIUS} Å")
print(f"  Minimum site distance: {MIN_SITE_DISTANCE} Å")
print("=" * 80)

SPATIAL EXCLUSION DOCKING CONFIGURATION
EquiBind device: cuda

Output directories:
  Conformer docking:     /home/manndo/master_dev/equibind_conformer_poses
  Spatial exclusion:     /home/manndo/master_dev/equibind_spatial_exclusion_poses

RDKit random seeds: [42, 123, 456, 789, 1001, 2022, 3141, 5926, 8675, 9999]
Conformers per seed: 20
Total conformers per batch: 200

Spatial exclusion settings:
  Binding sites to discover: 10
  Maximum conformers to try per site: 90
  Exclusion radius: 5.0 Å
  Minimum site distance: 3.0 Å


In [ ]:
# ============================================================================
# SPATIAL EXCLUSION HELPER FUNCTIONS
# ============================================================================
from dataclasses import dataclass, field
from typing import Tuple, List, Optional
from pathlib import Path
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

# Configuration for poses per binding site
POSES_PER_SITE = 20  # Number of different poses to find within each binding site
SITE_INCLUSION_RADIUS = 6.0  # Radius (Å) within which poses are considered "in the same site"
MIN_POSE_RMSD = 0.5  # Minimum RMSD (Å) between poses within a site for diversity

@dataclass
class ExclusionZone:
    """Represents a spherical exclusion zone around a found binding site."""
    center: Tuple[float, float, float]  # Centroid of the binding site
    radius: float  # Exclusion radius in Angstroms
    site_id: int  # Which binding site this represents
    pose_path: Path  # Path to the pose that defined this zone
    
    def contains_point(self, point: Tuple[float, float, float]) -> bool:
        """Check if a point (x, y, z) falls within this exclusion zone."""
        dx = point[0] - self.center[0]
        dy = point[1] - self.center[1]
        dz = point[2] - self.center[2]
        distance = np.sqrt(dx**2 + dy**2 + dz**2)
        return distance < self.radius
    
    def distance_to_point(self, point: Tuple[float, float, float]) -> float:
        """Calculate distance from point to the center of exclusion zone."""
        dx = point[0] - self.center[0]
        dy = point[1] - self.center[1]
        dz = point[2] - self.center[2]
        return np.sqrt(dx**2 + dy**2 + dz**2)


@dataclass
class PoseInSite:
    """Information about a single pose within a binding site."""
    pose_id: int  # Pose number within the site (1, 2, 3, ...)
    sdf_path: Path  # Path to the SDF file for this pose
    centroid: Tuple[float, float, float]  # Centroid of this pose
    conformer_id: int  # Which conformer was used
    seed_used: int  # Random seed used for conformer generation
    distance_to_site_center: float  # Distance from pose centroid to site center
    rmsd_to_reference: Optional[float] = None  # RMSD to the reference (first) pose
    
    def to_dict(self) -> dict:
        return {
            "pose_id": self.pose_id,
            "sdf_path": str(self.sdf_path),
            "centroid": list(self.centroid),
            "conformer_id": self.conformer_id,
            "seed_used": self.seed_used,
            "distance_to_site_center": self.distance_to_site_center,
            "rmsd_to_reference": self.rmsd_to_reference,
        }


@dataclass
class BindingSiteInfo:
    """Information about a found binding site with multiple poses."""
    site_id: int
    centroid: Tuple[float, float, float]  # Site center (from first/reference pose)
    sdf_path: Path  # Path to the reference SDF file for this site
    conformer_id: int  # Conformer that found this site
    attempt_number: int
    seed_used: int = 0  # Which random seed was used
    distance_to_nearest_site: Optional[float] = None
    poses: List[PoseInSite] = field(default_factory=list)  # Multiple poses within this site
    
    @property
    def num_poses(self) -> int:
        return len(self.poses)
    
    def to_dict(self) -> dict:
        return {
            "site_id": self.site_id,
            "centroid": list(self.centroid),
            "sdf_path": str(self.sdf_path),
            "conformer_id": self.conformer_id,
            "attempt_number": self.attempt_number,
            "seed_used": self.seed_used,
            "distance_to_nearest_site": self.distance_to_nearest_site,
            "num_poses": self.num_poses,
            "poses": [p.to_dict() for p in self.poses],
        }


@dataclass
class SpatialExclusionResult:
    """Results from spatial exclusion docking with multiple poses per site."""
    protein_name: str
    ligand_name: str
    protein_path: Path
    ligand_path: Path
    output_dir: Path
    binding_sites: List[BindingSiteInfo] = field(default_factory=list)
    exclusion_zones: List[ExclusionZone] = field(default_factory=list)
    total_conformers_tried: int = 0
    total_docking_attempts: int = 0
    rejected_in_exclusion: int = 0
    poses_rejected_outside_site: int = 0  # Poses that landed outside the target site
    poses_rejected_low_rmsd: int = 0  # Poses too similar to existing poses
    elapsed_time: float = 0.0
    status: str = "pending"
    error_message: str = ""
    seeds_used: List[int] = field(default_factory=list)
    
    @property
    def total_poses(self) -> int:
        return sum(site.num_poses for site in self.binding_sites)
    
    def to_dict(self) -> dict:
        return {
            "protein_name": self.protein_name,
            "ligand_name": self.ligand_name,
            "protein_path": str(self.protein_path),
            "ligand_path": str(self.ligand_path),
            "output_dir": str(self.output_dir),
            "status": self.status,
            "binding_sites": [s.to_dict() for s in self.binding_sites],
            "num_sites_found": len(self.binding_sites),
            "total_poses": self.total_poses,
            "total_conformers_tried": self.total_conformers_tried,
            "total_docking_attempts": self.total_docking_attempts,
            "rejected_in_exclusion": self.rejected_in_exclusion,
            "poses_rejected_outside_site": self.poses_rejected_outside_site,
            "poses_rejected_low_rmsd": self.poses_rejected_low_rmsd,
            "elapsed_time": self.elapsed_time,
            "error_message": self.error_message,
            "seeds_used": self.seeds_used,
        }


def get_ligand_centroid(mol) -> Optional[Tuple[float, float, float]]:
    """Calculate the centroid of a ligand molecule."""
    try:
        conf = mol.GetConformer()
        positions = conf.GetPositions()
        centroid = positions.mean(axis=0)
        return tuple(centroid)
    except Exception as e:
        print(f"Error calculating centroid: {e}")
        return None


def is_in_any_exclusion_zone(centroid: Tuple[float, float, float], 
                              exclusion_zones: List[ExclusionZone]) -> Tuple[bool, Optional[float]]:
    """
    Check if a centroid point falls within any exclusion zone.
    
    Args:
        centroid: (x, y, z) coordinates of the ligand centroid
        exclusion_zones: List of ExclusionZone objects
        
    Returns:
        Tuple of (is_excluded, distance_to_nearest_zone_center)
    """
    if not exclusion_zones:
        return False, None
    
    min_distance = float('inf')
    is_excluded = False
    
    for zone in exclusion_zones:
        dist = zone.distance_to_point(centroid)
        if dist < min_distance:
            min_distance = dist
        if zone.contains_point(centroid):
            is_excluded = True
    
    return is_excluded, min_distance if min_distance != float('inf') else None


def generate_diverse_conformers_batch(
    ligand_file: Path, 
    seeds: List[int] = None,
    conformers_per_seed: int = None,
) -> List[Tuple[Path, int, int]]:
    """
    Generate diverse conformers using MULTIPLE random seeds for maximum diversity.
    
    Each seed produces a different set of starting geometries, so using multiple
    seeds ensures we explore more of the conformational space.
    
    Args:
        ligand_file: Path to ligand file (SDF/MOL2/PDB)
        seeds: List of random seeds to use (default: RDKIT_RANDOM_SEEDS)
        conformers_per_seed: Number of conformers per seed (default: CONFORMERS_PER_SEED)
        
    Returns:
        List of (conformer_sdf_path, conformer_id, seed_used) tuples
    """
    # Use global defaults if not provided
    if seeds is None:
        seeds = RDKIT_RANDOM_SEEDS
    if conformers_per_seed is None:
        conformers_per_seed = CONFORMERS_PER_SEED
    
    # Load molecule
    suffix = ligand_file.suffix.lower()
    try:
        if suffix == ".sdf":
            suppl = Chem.SDMolSupplier(str(ligand_file), removeHs=False)
            mol = next(iter(suppl), None)
        elif suffix == ".mol2":
            mol = Chem.MolFromMol2File(str(ligand_file), removeHs=False)
        elif suffix == ".pdb":
            mol = Chem.MolFromPDBFile(str(ligand_file), removeHs=False)
        else:
            print(f"  Warning: Unsupported file format {suffix}")
            return [(ligand_file, 0, 0)]
        
        if mol is None:
            print(f"  Warning: Could not load molecule from {ligand_file}")
            return [(ligand_file, 0, 0)]
    except Exception as e:
        print(f"  Error loading molecule: {e}")
        return [(ligand_file, 0, 0)]
    
    # Add hydrogens
    mol_h = Chem.AddHs(mol)
    
    # Create output directory for conformers
    conformer_dir = ligand_file.parent / f"{ligand_file.stem}_conformers"
    conformer_dir.mkdir(parents=True, exist_ok=True)
    
    conformer_files = []
    global_conf_id = 0
    
    print(f"  Generating conformers with {len(seeds)} different random seeds...")
    print(f"  Seeds: {seeds}")
    print(f"  Conformers per seed: {conformers_per_seed}")
    
    for seed_idx, seed in enumerate(seeds):
        # Set up ETKDG parameters for this seed
        params = AllChem.ETKDGv3()
        params.randomSeed = seed
        params.numThreads = 0  # Use all available threads
        params.useRandomCoords = True
        params.maxIterations = 500
        
        # Generate conformers with this seed
        try:
            conf_ids = AllChem.EmbedMultipleConfs(mol_h, numConfs=conformers_per_seed, params=params)
        except Exception as e:
            print(f"    Seed {seed}: Failed to generate conformers: {e}")
            continue
        
        if len(conf_ids) == 0:
            # Try fallback with relaxed parameters
            params.enforceChirality = False
            try:
                conf_ids = AllChem.EmbedMultipleConfs(mol_h, numConfs=conformers_per_seed, params=params)
            except:
                pass
        
        if len(conf_ids) == 0:
            print(f"    Seed {seed}: No conformers generated")
            continue
        
        # Minimize conformers
        for conf_id in conf_ids:
            try:
                AllChem.MMFFOptimizeMolecule(mol_h, confId=conf_id, maxIters=200)
            except:
                pass
        
        # Save each conformer
        seed_confs_saved = 0
        for local_idx, conf_id in enumerate(conf_ids):
            global_conf_id += 1
            conf_path = conformer_dir / f"seed{seed}_conf{local_idx+1:02d}.sdf"
            try:
                writer = Chem.SDWriter(str(conf_path))
                writer.write(mol_h, confId=conf_id)
                writer.close()
                if conf_path.exists() and conf_path.stat().st_size > 0:
                    conformer_files.append((conf_path, global_conf_id, seed))
                    seed_confs_saved += 1
            except Exception as e:
                print(f"    Warning: Could not save conformer {global_conf_id}: {e}")
        
        print(f"    Seed {seed}: Generated {seed_confs_saved} conformers")
    
    if not conformer_files:
        print("  Warning: No conformers generated, using original ligand")
        return [(ligand_file, 0, 0)]
    
    print(f"  ✓ Total: {len(conformer_files)} diverse conformers from {len(seeds)} seeds")
    return conformer_files


print("✓ Spatial exclusion helper classes and functions defined.")
print("  - ExclusionZone: Spherical exclusion zone")
print("  - PoseInSite: Information about a single pose within a binding site")
print("  - BindingSiteInfo: Information about found binding sites (with multiple poses)")
print("  - SpatialExclusionResult: Results container with to_dict()")
print("  - is_in_any_exclusion_zone(): Check if point is excluded")
print("  - generate_diverse_conformers_batch(): Generate conformers with MULTIPLE seeds")
print()
print("New configuration parameters:")
print(f"  POSES_PER_SITE = {POSES_PER_SITE}")
print(f"  SITE_INCLUSION_RADIUS = {SITE_INCLUSION_RADIUS} Å")
print(f"  MIN_POSE_RMSD = {MIN_POSE_RMSD} Å")

✓ Spatial exclusion helper classes and functions defined.
  - ExclusionZone: Spherical exclusion zone
  - PoseInSite: Information about a single pose within a binding site
  - BindingSiteInfo: Information about found binding sites (with multiple poses)
  - SpatialExclusionResult: Results container with to_dict()
  - is_in_any_exclusion_zone(): Check if point is excluded
  - generate_diverse_conformers_batch(): Generate conformers with MULTIPLE seeds

New configuration parameters:
  POSES_PER_SITE = 20
  SITE_INCLUSION_RADIUS = 6.0 Å
  MIN_POSE_RMSD = 0.5 Å


In [ ]:
# ============================================================================
# MAIN SPATIAL EXCLUSION DOCKING FUNCTION (with Multiple Poses per Site)
# ============================================================================

def is_pose_diverse_from_existing(
    new_sdf: Path, 
    existing_poses: List[PoseInSite], 
    min_rmsd: float = MIN_POSE_RMSD
) -> Tuple[bool, Optional[float]]:
    """
    Check if a new pose is sufficiently different from existing poses in the site.
    
    Args:
        new_sdf: Path to the new pose SDF file
        existing_poses: List of existing PoseInSite objects
        min_rmsd: Minimum RMSD threshold for diversity
        
    Returns:
        Tuple of (is_diverse, min_rmsd_to_existing)
    """
    if not existing_poses:
        return True, None
    
    min_rmsd_found = float('inf')
    for pose in existing_poses:
        rmsd = compute_pose_rmsd(new_sdf, pose.sdf_path)
        if rmsd is not None and rmsd < min_rmsd_found:
            min_rmsd_found = rmsd
    
    if min_rmsd_found == float('inf'):
        return True, None
    
    return min_rmsd_found >= min_rmsd, min_rmsd_found


def load_existing_result(output_dir: Path) -> Optional[SpatialExclusionResult]:
    """
    Load an existing SpatialExclusionResult from a metadata JSON file.
    
    Args:
        output_dir: Directory containing spatial_exclusion_metadata.json
        
    Returns:
        SpatialExclusionResult if found and valid, None otherwise
    """
    metadata_path = output_dir / "spatial_exclusion_metadata.json"
    
    if not metadata_path.exists():
        return None
    
    try:
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
        
        # Reconstruct binding sites with poses
        binding_sites = []
        for site_data in metadata.get("binding_sites", []):
            poses = []
            for pose_data in site_data.get("poses", []):
                pose = PoseInSite(
                    pose_id=pose_data["pose_id"],
                    sdf_path=Path(pose_data["sdf_path"]),
                    centroid=tuple(pose_data["centroid"]),
                    conformer_id=pose_data["conformer_id"],
                    seed_used=pose_data["seed_used"],
                    distance_to_site_center=pose_data["distance_to_site_center"],
                    rmsd_to_reference=pose_data.get("rmsd_to_reference"),
                )
                # Verify pose file exists
                if pose.sdf_path.exists():
                    poses.append(pose)
            
            if poses:  # Only add site if it has valid poses
                site = BindingSiteInfo(
                    site_id=site_data["site_id"],
                    centroid=tuple(site_data["centroid"]),
                    sdf_path=Path(site_data["sdf_path"]),
                    conformer_id=site_data["conformer_id"],
                    attempt_number=site_data["attempt_number"],
                    seed_used=site_data["seed_used"],
                    distance_to_nearest_site=site_data.get("distance_to_nearest_site"),
                    poses=poses,
                )
                binding_sites.append(site)
        
        # If no valid binding sites found, return None
        if not binding_sites:
            return None
        
        # Reconstruct result
        result = SpatialExclusionResult(
            protein_name=metadata["protein_name"],
            ligand_name=metadata["ligand_name"],
            protein_path=Path(metadata["protein_path"]),
            ligand_path=Path(metadata["ligand_path"]),
            output_dir=Path(metadata["output_dir"]),
            binding_sites=binding_sites,
            total_conformers_tried=metadata.get("total_conformers_tried", 0),
            total_docking_attempts=metadata.get("total_docking_attempts", 0),
            rejected_in_exclusion=metadata.get("rejected_in_exclusion", 0),
            poses_rejected_outside_site=metadata.get("poses_rejected_outside_site", 0),
            poses_rejected_low_rmsd=metadata.get("poses_rejected_low_rmsd", 0),
            elapsed_time=metadata.get("elapsed_time", 0.0),
            status=metadata.get("status", "success"),
            error_message=metadata.get("error_message", ""),
            seeds_used=metadata.get("seeds_used", []),
        )
        
        return result
        
    except Exception as e:
        print(f"  Warning: Could not load existing result from {metadata_path}: {e}")
        return None


def dock_with_spatial_exclusion(
    protein_pdb: Path,
    ligand_file: Path,
    num_binding_sites: int = NUM_BINDING_SITES,
    poses_per_site: int = POSES_PER_SITE,
    conformers_per_seed: int = CONFORMERS_PER_SEED,
    exclusion_radius: float = EXCLUSION_RADIUS,
    site_inclusion_radius: float = SITE_INCLUSION_RADIUS,
    min_site_distance: float = MIN_SITE_DISTANCE,
    min_pose_rmsd: float = MIN_POSE_RMSD,
    max_conformers_per_site: int = MAX_CONFORMERS_PER_SITE,
    device: str = EQUIBIND_DEVICE,
    seeds: List[int] = None,
    skip_existing: bool = True,
) -> SpatialExclusionResult:
    """
    Discover multiple distinct binding sites AND multiple poses within each site.
    
    Algorithm:
    1. Generate conformers using MULTIPLE random seeds for diversity
    2. For each binding site to find:
       a. Find the first pose that defines the site (outside all exclusion zones)
       b. Then find additional diverse poses WITHIN that site:
          - Dock more conformers
          - Accept poses that land within site_inclusion_radius of site center
          - Reject poses too similar to existing poses (RMSD < min_pose_rmsd)
       c. Create exclusion zone around the site center
       d. Move to next site
    3. Repeat until all sites found or conformers exhausted
    
    Args:
        protein_pdb: Path to protein PDB file
        ligand_file: Path to ligand file (SDF/MOL2/PDB)
        num_binding_sites: Number of distinct binding sites to discover
        poses_per_site: Number of different poses to find within each binding site
        conformers_per_seed: Number of conformers to generate per random seed
        exclusion_radius: Radius (Å) to exclude around found sites (for finding NEW sites)
        site_inclusion_radius: Radius (Å) within which poses are considered "in the same site"
        min_site_distance: Minimum distance (Å) between site centroids
        min_pose_rmsd: Minimum RMSD (Å) between poses within a site for diversity
        max_conformers_per_site: Max conformers before giving up on a site
        device: EquiBind device (cpu/cuda)
        seeds: List of random seeds for conformer generation (default: RDKIT_RANDOM_SEEDS)
        skip_existing: If True, skip docking if results already exist (default: True)
    
    Returns:
        SpatialExclusionResult with all discovered binding sites and poses
    """
    # Use global seeds if not provided
    if seeds is None:
        seeds = RDKIT_RANDOM_SEEDS
    
    protein_name = get_file_stem(protein_pdb)
    ligand_name = get_file_stem(ligand_file)
    combo_name = f"{ligand_name}__{protein_name}_spatial"
    
    # Set up directories
    batch_dir = EQUIBIND_BATCH_DIR / "spatial_exclusion" / combo_name
    output_dir = SPATIAL_EXCLUSION_OUTPUT_DIR / f"{combo_name}_sites"
    
    # Check if results already exist
    if skip_existing:
        existing_result = load_existing_result(output_dir)
        if existing_result is not None:
            print(f"\n{'='*80}")
            print(f"SKIPPING (results exist): {combo_name}")
            print(f"{'='*80}")
            print(f"  Status: {existing_result.status}")
            print(f"  Binding sites: {len(existing_result.binding_sites)}")
            print(f"  Total poses: {existing_result.total_poses}")
            print(f"  Output: {existing_result.output_dir}")
            return existing_result
    
    # Create directories
    batch_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Prepare protein
    prepared_protein = prepare_protein_with_reduce(protein_pdb, batch_dir)
    
    # Initialize result
    result = SpatialExclusionResult(
        protein_name=protein_name,
        ligand_name=ligand_name,
        protein_path=protein_pdb,
        ligand_path=ligand_file,
        output_dir=output_dir,
        seeds_used=seeds.copy(),
    )
    
    start_time = time.time()
    exclusion_zones: List[ExclusionZone] = []
    binding_sites: List[BindingSiteInfo] = []
    
    print(f"\n{'='*80}")
    print(f"SPATIAL EXCLUSION DOCKING (Multi-Pose): {combo_name}")
    print(f"{'='*80}")
    print(f"  Target binding sites: {num_binding_sites}")
    print(f"  Poses per site: {poses_per_site}")
    print(f"  Exclusion radius (between sites): {exclusion_radius} Å")
    print(f"  Site inclusion radius (for poses): {site_inclusion_radius} Å")
    print(f"  Minimum site distance: {min_site_distance} Å")
    print(f"  Minimum pose RMSD: {min_pose_rmsd} Å")
    print(f"  Random seeds: {seeds}")
    print(f"  Conformers per seed: {conformers_per_seed}")
    print()
    
    # Generate ALL conformers upfront using multiple seeds
    print(f"  Generating diverse conformers with {len(seeds)} random seeds...")
    conformer_batch = generate_diverse_conformers_batch(
        ligand_file,
        seeds=seeds,
        conformers_per_seed=conformers_per_seed,
    )
    total_conformers = len(conformer_batch)
    print(f"  Total conformers available: {total_conformers}")
    
    # Track which conformers we've used
    conformer_idx = 0
    
    # Search for each binding site
    for site_num in range(1, num_binding_sites + 1):
        print(f"\n  {'─'*60}")
        print(f"  Searching for Binding Site {site_num}/{num_binding_sites}")
        print(f"  {'─'*60}")
        
        if exclusion_zones:
            print(f"  Active exclusion zones: {len(exclusion_zones)}")
            for ez in exclusion_zones:
                print(f"    - Site {ez.site_id}: center ({ez.center[0]:.1f}, {ez.center[1]:.1f}, {ez.center[2]:.1f}), r={ez.radius}Å")
        
        site_found = False
        site_center = None
        site_poses: List[PoseInSite] = []
        conformers_tried_for_site = 0
        reference_sdf = None
        
        # PHASE 1: Find the first pose that defines this binding site
        print(f"\n  Phase 1: Finding site location...")
        
        while conformer_idx < total_conformers and not site_found:
            if conformers_tried_for_site >= max_conformers_per_site:
                print(f"  ⚠️  Max conformers ({max_conformers_per_site}) reached for site {site_num}")
                break
            
            conf_path, conf_id, seed_used = conformer_batch[conformer_idx]
            conformer_idx += 1
            conformers_tried_for_site += 1
            result.total_conformers_tried += 1
            
            # Create EquiBind output directory for this attempt
            equibind_out = batch_dir / f"site{site_num}_conf{conf_id:03d}_seed{seed_used}"
            if equibind_out.exists():
                shutil.rmtree(equibind_out)
            equibind_out.mkdir(parents=True, exist_ok=True)
            
            # Run EquiBind (in-process, using already-loaded GPU model)
            success, sdf_path, error = _run_equibind(
                protein_pdb=prepared_protein,
                ligand_file=conf_path,
                output_dir=equibind_out,
                seed=seed_used,
                device=device,
            )
            
            result.total_docking_attempts += 1
            
            if not success or sdf_path is None:
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Compute centroid of docked pose
            centroid = compute_centroid(sdf_path)
            if centroid is None:
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Check if centroid is in any exclusion zone
            is_excluded, dist_to_nearest = is_in_any_exclusion_zone(centroid, exclusion_zones)
            
            if is_excluded:
                result.rejected_in_exclusion += 1
                print(f"    Conf {conf_id} (seed {seed_used}): REJECTED (in exclusion zone, {dist_to_nearest:.1f}Å from nearest site)")
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Check minimum distance to existing sites
            if dist_to_nearest is not None and dist_to_nearest < min_site_distance:
                result.rejected_in_exclusion += 1
                print(f"    Conf {conf_id} (seed {seed_used}): REJECTED (too close: {dist_to_nearest:.1f}Å < {min_site_distance}Å)")
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # SUCCESS! Found a new binding site
            site_found = True
            site_center = centroid
            
            # Save reference pose to output directory
            site_dir = output_dir / f"site_{site_num:02d}"
            site_dir.mkdir(parents=True, exist_ok=True)
            reference_sdf = site_dir / f"pose_01.sdf"
            shutil.copy2(sdf_path, reference_sdf)
            
            # Create first pose info
            first_pose = PoseInSite(
                pose_id=1,
                sdf_path=reference_sdf,
                centroid=centroid,
                conformer_id=conf_id,
                seed_used=seed_used,
                distance_to_site_center=0.0,
                rmsd_to_reference=0.0,
            )
            site_poses.append(first_pose)
            
            print(f"    ✓ BINDING SITE {site_num} FOUND!")
            print(f"      Site center: ({centroid[0]:.2f}, {centroid[1]:.2f}, {centroid[2]:.2f})")
            print(f"      Reference pose: conformer {conf_id} (seed {seed_used})")
            
            # Clean up equibind output
            try:
                shutil.rmtree(equibind_out)
            except:
                pass
        
        if not site_found:
            print(f"  ✗ Could not find binding site {site_num}")
            if conformer_idx >= total_conformers:
                result.error_message = f"Exhausted all {total_conformers} conformers. Found {len(binding_sites)}/{num_binding_sites} sites"
            break
        
        # PHASE 2: Find additional poses within this binding site
        print(f"\n  Phase 2: Finding additional poses within site {site_num}...")
        print(f"    Target: {poses_per_site} poses, currently have: {len(site_poses)}")
        
        max_attempts_for_poses = min(max_conformers_per_site * 2, total_conformers - conformer_idx)
        attempts_for_poses = 0
        
        while len(site_poses) < poses_per_site and conformer_idx < total_conformers:
            if attempts_for_poses >= max_attempts_for_poses:
                print(f"    ⚠️  Max attempts ({max_attempts_for_poses}) reached for additional poses")
                break
            
            conf_path, conf_id, seed_used = conformer_batch[conformer_idx]
            conformer_idx += 1
            attempts_for_poses += 1
            result.total_conformers_tried += 1
            
            # Create EquiBind output directory
            equibind_out = batch_dir / f"site{site_num}_pose{len(site_poses)+1}_conf{conf_id:03d}"
            if equibind_out.exists():
                shutil.rmtree(equibind_out)
            equibind_out.mkdir(parents=True, exist_ok=True)
            
            # Run EquiBind (in-process, using already-loaded GPU model)
            success, sdf_path, error = _run_equibind(
                protein_pdb=prepared_protein,
                ligand_file=conf_path,
                output_dir=equibind_out,
                seed=seed_used,
                device=device,
            )
            
            result.total_docking_attempts += 1
            
            if not success or sdf_path is None:
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Compute centroid of docked pose
            centroid = compute_centroid(sdf_path)
            if centroid is None:
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Check if pose is WITHIN the current binding site
            dist_to_site = np.sqrt(
                (centroid[0] - site_center[0])**2 +
                (centroid[1] - site_center[1])**2 +
                (centroid[2] - site_center[2])**2
            )
            
            if dist_to_site > site_inclusion_radius:
                result.poses_rejected_outside_site += 1
                print(f"    Conf {conf_id}: REJECTED (outside site: {dist_to_site:.1f}Å > {site_inclusion_radius}Å)")
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Check if pose is diverse enough from existing poses
            is_diverse, min_rmsd_to_existing = is_pose_diverse_from_existing(
                sdf_path, site_poses, min_pose_rmsd
            )
            
            if not is_diverse:
                result.poses_rejected_low_rmsd += 1
                print(f"    Conf {conf_id}: REJECTED (too similar: RMSD {min_rmsd_to_existing:.2f}Å < {min_pose_rmsd}Å)")
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Calculate RMSD to reference pose
            rmsd_to_ref = compute_pose_rmsd(sdf_path, reference_sdf)
            
            # SUCCESS! Found a diverse pose within the site
            pose_num = len(site_poses) + 1
            pose_sdf = site_dir / f"pose_{pose_num:02d}.sdf"
            shutil.copy2(sdf_path, pose_sdf)
            
            new_pose = PoseInSite(
                pose_id=pose_num,
                sdf_path=pose_sdf,
                centroid=centroid,
                conformer_id=conf_id,
                seed_used=seed_used,
                distance_to_site_center=dist_to_site,
                rmsd_to_reference=rmsd_to_ref,
            )
            site_poses.append(new_pose)
            
            print(f"    ✓ Pose {pose_num}: conf {conf_id}, dist={dist_to_site:.1f}Å, RMSD={rmsd_to_ref:.2f}Å")
            
            # Clean up
            try:
                shutil.rmtree(equibind_out)
            except:
                pass
        
        # Create BindingSiteInfo with all poses
        site_info = BindingSiteInfo(
            site_id=site_num,
            centroid=site_center,
            sdf_path=reference_sdf,
            conformer_id=site_poses[0].conformer_id,
            attempt_number=conformers_tried_for_site,
            seed_used=site_poses[0].seed_used,
            distance_to_nearest_site=dist_to_nearest if 'dist_to_nearest' in dir() else None,
            poses=site_poses,
        )
        binding_sites.append(site_info)
        
        # Create exclusion zone for this site
        new_zone = ExclusionZone(
            center=site_center,
            radius=exclusion_radius,
            site_id=site_num,
            pose_path=reference_sdf,
        )
        exclusion_zones.append(new_zone)
        
        print(f"\n    Site {site_num} complete: {len(site_poses)}/{poses_per_site} poses found")
        print(f"      Exclusion zone created: radius {exclusion_radius}Å")
    
    # Finalize result
    result.binding_sites = binding_sites
    result.exclusion_zones = exclusion_zones
    result.elapsed_time = time.time() - start_time
    
    if len(binding_sites) >= num_binding_sites:
        result.status = "success"
    elif len(binding_sites) > 0:
        result.status = "partial"
    else:
        result.status = "failed"
    
    # Save metadata
    metadata = result.to_dict()
    metadata["exclusion_zones"] = [
        {
            "site_id": ez.site_id,
            "center": list(ez.center),
            "radius": ez.radius,
        }
        for ez in exclusion_zones
    ]
    metadata["conformer_generation"] = {
        "seeds_used": seeds,
        "conformers_per_seed": conformers_per_seed,
        "total_conformers_generated": total_conformers,
    }
    metadata["pose_parameters"] = {
        "poses_per_site": poses_per_site,
        "site_inclusion_radius": site_inclusion_radius,
        "min_pose_rmsd": min_pose_rmsd,
    }
    
    metadata_path = output_dir / "spatial_exclusion_metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Print summary
    print(f"\n{'='*80}")
    print(f"SPATIAL EXCLUSION RESULTS (Multi-Pose): {combo_name}")
    print(f"{'='*80}")
    print(f"  Status: {result.status}")
    print(f"  Binding sites found: {len(binding_sites)}/{num_binding_sites}")
    print(f"  Total poses found: {result.total_poses}")
    print(f"  Seeds used for conformers: {seeds}")
    print(f"  Total conformers available: {total_conformers}")
    print(f"  Total conformers tried: {result.total_conformers_tried}")
    print(f"  Total docking attempts: {result.total_docking_attempts}")
    print(f"  Poses rejected (in exclusion zone): {result.rejected_in_exclusion}")
    print(f"  Poses rejected (outside site): {result.poses_rejected_outside_site}")
    print(f"  Poses rejected (low RMSD): {result.poses_rejected_low_rmsd}")
    print(f"  Elapsed time: {result.elapsed_time:.1f}s")
    print(f"  Output directory: {output_dir}")
    
    if binding_sites:
        print(f"\n  Binding Sites Summary:")
        for site in binding_sites:
            print(f"    Site {site.site_id}: {site.num_poses} poses")
            for pose in site.poses:
                print(f"      - Pose {pose.pose_id}: conf={pose.conformer_id}, seed={pose.seed_used}, "
                      f"dist={pose.distance_to_site_center:.1f}Å, RMSD={pose.rmsd_to_reference:.2f}Å")
    
    return result


print("✓ Main spatial exclusion docking function defined (with multi-pose support).")
print("✓ load_existing_result() function added for loading cached results.")
print("\nUsage:")
print("  result = dock_with_spatial_exclusion(")
print("      protein_pdb,")
print("      ligand_file,")
print("      num_binding_sites=3,        # Find 3 distinct sites")
print("      poses_per_site=5,           # 5 different poses per site")
print("      conformers_per_seed=5,      # 5 conformers per seed")
print("      exclusion_radius=8.0,       # 8Å exclusion around each site")
print("      site_inclusion_radius=6.0,  # Poses within 6Å of site center")
print("      min_pose_rmsd=1.5,          # Minimum RMSD between poses")
print("      seeds=[42, 123, 456, ...],  # Multiple seeds for diversity")
print("      skip_existing=True,         # Skip if results already exist")
print("  )")

✓ Main spatial exclusion docking function defined (with multi-pose support).
✓ load_existing_result() function added for loading cached results.

Usage:
  result = dock_with_spatial_exclusion(
      protein_pdb,
      ligand_file,
      num_binding_sites=3,        # Find 3 distinct sites
      poses_per_site=5,           # 5 different poses per site
      conformers_per_seed=5,      # 5 conformers per seed
      exclusion_radius=8.0,       # 8Å exclusion around each site
      site_inclusion_radius=6.0,  # Poses within 6Å of site center
      min_pose_rmsd=1.5,          # Minimum RMSD between poses
      seeds=[42, 123, 456, ...],  # Multiple seeds for diversity
      skip_existing=True,         # Skip if results already exist
  )


In [ ]:
# ============================================================================
# BATCH SPATIAL EXCLUSION DOCKING (with Multi-Pose Support)
# ============================================================================

def run_spatial_exclusion_batch(
    proteins: List[Path],
    ligands: List[Path],
    num_binding_sites: int = NUM_BINDING_SITES,
    poses_per_site: int = POSES_PER_SITE,
    conformers_per_seed: int = CONFORMERS_PER_SEED,
    exclusion_radius: float = EXCLUSION_RADIUS,
    site_inclusion_radius: float = SITE_INCLUSION_RADIUS,
    min_site_distance: float = MIN_SITE_DISTANCE,
    min_pose_rmsd: float = MIN_POSE_RMSD,
    device: str = EQUIBIND_DEVICE,
    skip_existing: bool = True,
) -> List[SpatialExclusionResult]:
    """
    Run spatial exclusion docking for all protein-ligand combinations.
    
    Args:
        proteins: List of protein PDB paths
        ligands: List of ligand file paths
        num_binding_sites: Number of distinct binding sites to find per combination
        poses_per_site: Number of different poses to find within each binding site
        conformers_per_seed: Number of conformers to generate per random seed
        exclusion_radius: Radius (Å) of exclusion zones (between sites)
        site_inclusion_radius: Radius (Å) within which poses are "in the same site"
        min_site_distance: Minimum distance (Å) between sites
        min_pose_rmsd: Minimum RMSD (Å) between poses within a site
        device: EquiBind device (cpu/cuda)
        skip_existing: If True, skip combinations with existing results (default: True)
    
    Returns:
        List of SpatialExclusionResult objects
    """
    total_combinations = len(proteins) * len(ligands)
    
    print("=" * 80)
    print("SPATIAL EXCLUSION BATCH DOCKING (Multi-Pose)")
    print("=" * 80)
    print(f"Proteins: {len(proteins)}")
    print(f"Ligands: {len(ligands)}")
    print(f"Total combinations: {total_combinations}")
    print(f"Binding sites per combination: {num_binding_sites}")
    print(f"Poses per site: {poses_per_site}")
    print(f"Conformers per seed: {conformers_per_seed}")
    print(f"Exclusion radius: {exclusion_radius} Å")
    print(f"Site inclusion radius: {site_inclusion_radius} Å")
    print(f"Min pose RMSD: {min_pose_rmsd} Å")
    print(f"Skip existing: {skip_existing}")
    print(f"Expected total binding sites: {total_combinations * num_binding_sites}")
    print(f"Expected total poses: {total_combinations * num_binding_sites * poses_per_site}")
    print()
    
    results: List[SpatialExclusionResult] = []
    skipped_count = 0
    new_count = 0
    
    for idx, (protein, ligand) in enumerate(itertools.product(proteins, ligands), 1):
        print(f"\n[{idx}/{total_combinations}] Processing:")
        print(f"  Protein: {protein.name}")
        print(f"  Ligand: {ligand.name}")
        
        result = dock_with_spatial_exclusion(
            protein_pdb=protein,
            ligand_file=ligand,
            num_binding_sites=num_binding_sites,
            poses_per_site=poses_per_site,
            conformers_per_seed=conformers_per_seed,
            exclusion_radius=exclusion_radius,
            site_inclusion_radius=site_inclusion_radius,
            min_site_distance=min_site_distance,
            min_pose_rmsd=min_pose_rmsd,
            device=device,
            skip_existing=skip_existing,
        )
        
        results.append(result)
        
        # Check if this was skipped (elapsed_time would be from previous run if loaded)
        # A newly run result will have elapsed_time > 0 and was just computed
        if result.elapsed_time == 0 or (skip_existing and result.status in ["success", "partial"]):
            skipped_count += 1
        else:
            new_count += 1
        
        status_icon = {
            "success": "✓",
            "partial": "◐", 
            "failed": "✗",
        }.get(result.status, "?")
        
        print(f"\n  {status_icon} SUMMARY: {result.status.upper()}")
        print(f"    Sites found: {len(result.binding_sites)}/{num_binding_sites}")
        print(f"    Total poses: {result.total_poses}")
    
    # Print overall batch summary
    print("\n" + "=" * 80)
    print("BATCH SUMMARY")
    print("=" * 80)
    
    successful = sum(1 for r in results if r.status == "success")
    partial = sum(1 for r in results if r.status == "partial")
    failed = sum(1 for r in results if r.status == "failed")
    total_sites = sum(len(r.binding_sites) for r in results)
    total_poses = sum(r.total_poses for r in results)
    total_time = sum(r.elapsed_time for r in results)
    
    print(f"  Successful: {successful}/{total_combinations}")
    print(f"  Partial: {partial}/{total_combinations}")
    print(f"  Failed: {failed}/{total_combinations}")
    print(f"  Total binding sites found: {total_sites}")
    print(f"  Total poses found: {total_poses}")
    print(f"  Total time (new runs): {total_time:.1f}s ({total_time/60:.1f} min)")
    
    # Save batch summary
    summary_data = {
        "timestamp": datetime.now().isoformat(),
        "configuration": {
            "num_binding_sites": num_binding_sites,
            "poses_per_site": poses_per_site,
            "conformers_per_seed": conformers_per_seed,
            "exclusion_radius": exclusion_radius,
            "site_inclusion_radius": site_inclusion_radius,
            "min_site_distance": min_site_distance,
            "min_pose_rmsd": min_pose_rmsd,
            "skip_existing": skip_existing,
        },
        "results": [r.to_dict() for r in results],
    }
    
    summary_path = EQUIBIND_OUTPUT_DIR / "spatial_exclusion_batch_summary.json"
    with open(summary_path, 'w') as f:
        json.dump(summary_data, f, indent=2)
    print(f"\n  Batch summary saved to: {summary_path}")
    
    return results


print("✓ Batch spatial exclusion function defined (with multi-pose support and skip_existing).")

✓ Batch spatial exclusion function defined (with multi-pose support and skip_existing).


In [ ]:
def setup_equibind_batch_directory(
    protein_pdb: Path,
    ligand_file: Path,
    batch_base_dir: Path,
) -> Tuple[Path, Path, Path]:
    """
    Set up batch directory structure for EquiBind docking.
    
    Creates directory structure like:
    equibind_batches/sdf/LigandName__ProteinName/
        ├── ProteinName_protein.pdb
        └── LigandName_ligand.sdf
    
    Returns: (batch_dir, prepared_protein_path, prepared_ligand_path)
    """
    protein_name = get_file_stem(protein_pdb)
    ligand_name = get_file_stem(ligand_file)
    ligand_ext = ligand_file.suffix.lower().lstrip('.')  # 'sdf', 'mol2', or 'pdb'
    
    combo_name = f"{ligand_name}__{protein_name}"
    batch_dir = batch_base_dir / ligand_ext / combo_name
    batch_dir.mkdir(parents=True, exist_ok=True)
    
    # Prepare protein file (with reduce if available)
    protein_dest = batch_dir / f"{protein_name}_protein.pdb"
    if not protein_dest.exists():
        protein_dest = prepare_protein_with_reduce(protein_pdb, batch_dir)
        expected_name = batch_dir / f"{protein_name}_protein.pdb"
        if protein_dest != expected_name and protein_dest.exists():
            shutil.move(protein_dest, expected_name)
            protein_dest = expected_name
    
    # Prepare ligand file
    ligand_dest = batch_dir / f"{ligand_name}_ligand{ligand_file.suffix}"
    if not ligand_dest.exists():
        shutil.copy2(ligand_file, ligand_dest)
    
    return batch_dir, protein_dest, ligand_dest


def dock_protein_ligand_multiple_poses(
    protein_pdb: Path,
    ligand_file: Path,
    num_poses: int = NUM_POSES,
    num_conformers: int = NUM_CONFORMERS,
    max_attempts: int = MAX_ATTEMPTS,
    rmsd_threshold: float = POSE_RMSD_THRESHOLD,
    device: str = EQUIBIND_DEVICE,
) -> DockingResult:
    """
    Dock a protein-ligand pair and generate multiple diverse poses using RDKit conformers.
    
    Strategy to generate diverse poses:
    1. Generate multiple 3D conformers using RDKit's EmbedMultipleConfs()
    2. Run EquiBind on each conformer independently
    3. Track pose RMSD to ensure diversity
    4. Save each unique pose with a distinct ID
    """
    protein_name = get_file_stem(protein_pdb)
    ligand_name = get_file_stem(ligand_file)
    combo_name = f"{ligand_name}__{protein_name}"
    
    # Set up directories
    batch_dir, prepared_protein, prepared_ligand = setup_equibind_batch_directory(
        protein_pdb, ligand_file, EQUIBIND_BATCH_DIR
    )
    # Use CONFORMER_DOCKING_OUTPUT_DIR if defined, otherwise fall back to EQUIBIND_OUTPUT_DIR
    try:
        output_dir = CONFORMER_DOCKING_OUTPUT_DIR / combo_name
    except NameError:
        output_dir = EQUIBIND_OUTPUT_DIR / combo_name
    conformers_dir = batch_dir / "conformers"
    conformers_dir.mkdir(parents=True, exist_ok=True)
    
    result = DockingResult(
        protein_name=protein_name,
        ligand_name=ligand_name,
        protein_path=protein_pdb,
        ligand_path=ligand_file,
        output_dir=output_dir,
        status="pending",
    )
    
    start_time = time.time()
    
    # Check if we should skip (existing results)
    if not OVERWRITE_EXISTING and output_dir.exists():
        existing_sdfs = list(output_dir.glob("pose_*.sdf"))
        if len(existing_sdfs) >= num_poses:
            result.status = "skipped"
            for i, sdf in enumerate(sorted(existing_sdfs)[:num_poses]):
                centroid = compute_centroid(sdf) or (0.0, 0.0, 0.0)
                result.poses.append(PoseInfo(
                    pose_id=i + 1,
                    sdf_path=sdf,
                    centroid=centroid,
                    seed=0,
                    conformer_id=0,
                ))
            result.elapsed_time = time.time() - start_time
            return result
    
    # Create/clean output directory for final poses
    output_dir.mkdir(parents=True, exist_ok=True)
    
    poses: List[PoseInfo] = []
    attempts = 0
    
    print(f"  Docking {combo_name}...")
    print(f"    Batch dir: {batch_dir}")
    
    # Step 1: Generate RDKit conformers
    print(f"    Generating {num_conformers} RDKit conformers...")
    mol, conf_ids = generate_rdkit_conformers(
        ligand_file, 
        num_conformers=num_conformers,
        random_seed=RDKIT_RANDOM_SEED,
        prune_rms_thresh=RDKIT_PRUNE_RMS_THRESH,
    )
    
    result.num_conformers_generated = len(conf_ids) if conf_ids else 0
    
    if mol is None or not conf_ids:
        print(f"    ⚠️  Could not generate RDKit conformers, falling back to original ligand")
        # Fallback: use the original ligand file
        conformer_files = [prepared_ligand]
    else:
        print(f"    ✓ Generated {len(conf_ids)} diverse conformers")
        
        # Save each conformer as a separate SDF file
        conformer_files = []
        for i, conf_id in enumerate(conf_ids):
            conf_sdf = conformers_dir / f"conformer_{i+1:02d}.sdf"
            if save_conformer_to_sdf(mol, conf_id, conf_sdf):
                conformer_files.append(conf_sdf)
        
        if not conformer_files:
            print(f"    ⚠️  Failed to save conformers, using original ligand")
            conformer_files = [prepared_ligand]
    
    # Step 2: Dock each conformer with EquiBind
    print(f"    Running EquiBind on {len(conformer_files)} conformers...")
    
    for conf_idx, conf_file in enumerate(conformer_files):
        if len(poses) >= num_poses:
            break
        
        attempts += 1
        
        # Create equibind output directory for this conformer
        equibind_out = batch_dir / f"equibind_conf_{conf_idx+1:02d}"
        if equibind_out.exists():
            shutil.rmtree(equibind_out)
        equibind_out.mkdir(parents=True, exist_ok=True)
        
        # Run EquiBind on this conformer (in-process, using already-loaded GPU model)
        success, sdf_path, error = _run_equibind(
            protein_pdb=prepared_protein,
            ligand_file=conf_file,
            output_dir=equibind_out,
            seed=RDKIT_RANDOM_SEED,  # Use consistent seed since diversity comes from conformers
            device=device,
        )
        
        if not success or sdf_path is None:
            if attempts == 1:
                print(f"    ⚠️  Conformer {conf_idx+1} docking failed: {error[:100]}")
            continue
        
        # Check if this pose is unique
        if is_pose_unique(sdf_path, poses, rmsd_threshold):
            pose_id = len(poses) + 1
            final_sdf = output_dir / f"pose_{pose_id:02d}.sdf"
            shutil.copy2(sdf_path, final_sdf)
            
            centroid = compute_centroid(final_sdf) or (0.0, 0.0, 0.0)
            pose_info = PoseInfo(
                pose_id=pose_id,
                sdf_path=final_sdf,
                centroid=centroid,
                seed=RDKIT_RANDOM_SEED,
                conformer_id=conf_idx + 1,
            )
            poses.append(pose_info)
            print(f"    ✓ Pose {pose_id}/{num_poses} from conformer {conf_idx+1}")
        else:
            print(f"    - Conformer {conf_idx+1}: Pose too similar to existing, skipping...")
        
        # Clean up attempt directory to save space
        try:
            shutil.rmtree(equibind_out)
        except:
            pass
    
    # Update result
    result.poses = poses
    result.num_attempts = attempts
    result.elapsed_time = time.time() - start_time
    
    if len(poses) >= num_poses:
        result.status = "success"
    elif len(poses) > 0:
        result.status = "partial"
        result.error_message = f"Only {len(poses)}/{num_poses} unique poses generated from {len(conformer_files)} conformers"
    else:
        result.status = "failed"
        if not result.error_message:
            result.error_message = "Could not generate any poses"
    
    # Save result metadata
    metadata_path = output_dir / "docking_metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(result.to_dict(), f, indent=2)
    
    return result


print("Multi-pose docking function with RDKit conformer generation defined.")

Multi-pose docking function with RDKit conformer generation defined.


In [ ]:
def run_equibind_multiligand(
    protein_pdb: Path,
    ligand_file: Path,
    output_dir: Path,
    seed: int = 1,
    device: str = "cpu",
) -> Tuple[bool, Optional[Path], str]:
    """
    Run EquiBind multiligand_inference.py for a protein with ligand file.
    
    Uses conda environment 'equibind' and runs the command:
    conda run -n equibind python ~/docking_tools/EquiBind/multiligand_inference.py \
      -o ./equibind_out \
      -r protein.pdb \
      -l ligand.sdf \
      --device cpu
    
    Returns: (success, output_sdf_path, error_message)
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Build the command - use conda run to execute in equibind environment
    cmd = [
        "conda", "run", "-n", "equibind", "--no-capture-output",
        "python", str(EQUIBIND_MULTILIGAND_SCRIPT),
        "-o", str(output_dir),
        "-r", str(protein_pdb),
        "-l", str(ligand_file),
        "--seed", str(seed),
        "--device", device,
    ]
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=600,  # 10 minute timeout per docking
            cwd=str(output_dir.parent),
        )
        
        # Check for output.sdf in output directory
        output_sdf = output_dir / "output.sdf"
        if output_sdf.exists() and output_sdf.stat().st_size > 0:
            return True, output_sdf, ""
        
        # Check alternative locations
        for sdf in output_dir.rglob("*.sdf"):
            if sdf.stat().st_size > 0 and "failed" not in sdf.name.lower():
                return True, sdf, ""
        
        error_msg = f"No output SDF found. stdout: {result.stdout[-500:] if result.stdout else ''} stderr: {result.stderr[-500:] if result.stderr else ''}"
        return False, None, error_msg
        
    except subprocess.TimeoutExpired:
        return False, None, "Docking timeout (600s)"
    except Exception as e:
        return False, None, str(e)


print("EquiBind execution function defined (using conda environment 'equibind').")

EquiBind execution function defined (using conda environment 'equibind').


In [ ]:
# ============================================================================
# RUN SPATIAL EXCLUSION DOCKING ON ALL PROTEIN-LIGAND COMBINATIONS
# ============================================================================

print("=" * 80)
print("RUNNING SPATIAL EXCLUSION DOCKING ON ALL COMBINATIONS")
print("=" * 80)
print(f"\nProteins: {len(receptor_files)}")
for p in receptor_files:
    print(f"  - {p.name}")
print(f"\nLigands: {len(ligand_files)}")
for l in ligand_files:
    print(f"  - {l.name}")
print(f"\nTotal combinations: {len(receptor_files) * len(ligand_files)}")
print(f"Binding sites per combination: {NUM_BINDING_SITES}")
print(f"Poses per site: {POSES_PER_SITE}")
print(f"Expected total poses: {len(receptor_files) * len(ligand_files) * NUM_BINDING_SITES * POSES_PER_SITE}")
print(f"\nSkip existing results: True (will load from cache if available)")
print()

# Run batch spatial exclusion docking
spatial_exclusion_results = run_spatial_exclusion_batch(
    proteins=receptor_files,
    ligands=ligand_files,
    num_binding_sites=NUM_BINDING_SITES,
    poses_per_site=POSES_PER_SITE,
    conformers_per_seed=CONFORMERS_PER_SEED,
    exclusion_radius=EXCLUSION_RADIUS,
    site_inclusion_radius=SITE_INCLUSION_RADIUS,
    min_site_distance=MIN_SITE_DISTANCE,
    min_pose_rmsd=MIN_POSE_RMSD,
    device=EQUIBIND_DEVICE,
    skip_existing=False,  # Skip combinations that already have results
)

print("\n" + "=" * 80)
print("SPATIAL EXCLUSION DOCKING COMPLETE")
print("=" * 80)

RUNNING SPATIAL EXCLUSION DOCKING ON ALL COMBINATIONS

Proteins: 8
  - Orai1WT-MDSnap-Fr300.pdb
  - Orai1WT-MDSnap-Fr300_cleaned.pdb
  - Orai1WT-MDSnap-Fr400.pdb
  - Orai1WT-MDSnap-Fr400_cleaned.pdb
  - Orai1WT-MDSnap-Fr499.pdb
  - Orai1WT-MDSnap-Fr499_cleaned.pdb
  - Orai1WT-START-Fr0.pdb
  - Orai1WT-START-Fr0_cleaned.pdb

Ligands: 5
  - 2abp-nh2-OPT.sdf
  - 2abp-nh3p-OPT.sdf
  - Synta-66-OPT-Singlet.sdf
  - gsk7975a-deprot-OPT.sdf
  - gsk7975a-prot-OPT.sdf

Total combinations: 40
Binding sites per combination: 10
Poses per site: 20
Expected total poses: 8000

Skip existing results: True (will load from cache if available)

SPATIAL EXCLUSION BATCH DOCKING (Multi-Pose)
Proteins: 8
Ligands: 5
Total combinations: 40
Binding sites per combination: 10
Poses per site: 20
Conformers per seed: 20
Exclusion radius: 5.0 Å
Site inclusion radius: 6.0 Å
Min pose RMSD: 0.5 Å
Skip existing: False
Expected total binding sites: 400
Expected total poses: 8000


[1/40] Processing:
  Protein: Orai1WT-MDS

# Results